# Leak-2019 temporal replication notebook

This notebook repeats the 2018 empirical protocol on the 2019 SCADA and leakage records. The assessment uses the same signal set and the same four baseline definitions; leakage labels are introduced only after score construction.

## Pump-feature construction

Construct the same pump-flow descriptors for the 2019 PUMP_1 signal.

In [2]:
import os
import numpy as np
import pandas as pd


input_file = "2019_SCADA_Flows.csv"
output_dir = "pump_features_2019"
output_file = os.path.join(output_dir, "PUMP_1_pump_features_2019.csv")
summary_file = os.path.join(output_dir, "PUMP_1_pump_features_2019_summary.csv")

sep = ";"
timestamp_col = "Timestamp"
pump_col = "PUMP_1"

# SCADA sampling: 5 minutes
steps_5min = 1
steps_30min = 6
steps_60min = 12

rolling_30min = 6
rolling_60min = 12

# Time-of-day reference: 5-minute slots, cyclic +/- 30 minutes
slots_per_day = 288
tod_neighbour_slots = 6  # 6 * 5 min = +/- 30 min

# Pump activity threshold.
# Derive inactive/active pump-flow indicator.
# The automatic threshold is intentionally conservative.
activity_quantile = 0.01
activity_min_threshold = 1e-9

os.makedirs(output_dir, exist_ok=True)

# ROBUST CSV LOADING
# Handles semicolon separated CSV and decimal comma / decimal point.

def read_semicolon_csv_with_decimal_flex(path, sep=";", timestamp_col="Timestamp"):
    df = pd.read_csv(path, sep=sep, dtype=str)

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="raise")

    for col in df.columns:
        if col == timestamp_col:
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values(timestamp_col).reset_index(drop=True)

flows = read_semicolon_csv_with_decimal_flex(
    input_file,
    sep=sep,
    timestamp_col=timestamp_col
)

if pump_col not in flows.columns:
    raise ValueError(
        f"Pump column '{pump_col}' not found in {input_file}. "
        f"Available columns: {list(flows.columns)}"
    )

# FEATURE CONSTRUCTION

df = flows[[timestamp_col, pump_col]].copy()
df = df.rename(columns={pump_col: "q_p"})

# Basic timestamp diagnostics
df["year"] = df[timestamp_col].dt.year
df["month"] = df[timestamp_col].dt.month
df["dayofyear"] = df[timestamp_col].dt.dayofyear
df["hour"] = df[timestamp_col].dt.hour
df["minute"] = df[timestamp_col].dt.minute
df["slot_5min"] = df["hour"] * 12 + (df["minute"] // 5)

# Native pump flow
# q_p(t)
df["q_p"] = df["q_p"].astype(float)

# Dynamic features
# Delta q_p,5(t) = q_p(t) - q_p(t-1)
# Delta q_p,30(t) = q_p(t) - q_p(t-6)
# Delta q_p,60(t) = q_p(t) - q_p(t-12)
df["delta_q_p_5min"] = df["q_p"].diff(steps_5min)
df["delta_q_p_30min"] = df["q_p"].diff(steps_30min)
df["delta_q_p_60min"] = df["q_p"].diff(steps_60min)

# Rolling variability features
# 5-minute rolling std is intentionally not used:
# with 5-minute sampling it would have too few observations to be stable.
df["std_q_p_30min"] = df["q_p"].rolling(
    window=rolling_30min,
    min_periods=rolling_30min
).std()

df["std_q_p_60min"] = df["q_p"].rolling(
    window=rolling_60min,
    min_periods=rolling_60min
).std()

# Pump activity indicator derived from measured pump flow
abs_q = df["q_p"].abs()
activity_threshold = max(activity_min_threshold, abs_q.quantile(activity_quantile))
df["pump_activity"] = (abs_q > activity_threshold).astype(int)

# TIME-OF-DAY REFERENCE PROFILE
# For each 5-minute slot, collect all observations from the full period
# within cyclic +/- 30 minutes around that slot, then take the median.

slot_reference = {}

for slot in range(slots_per_day):
    neighbour_slots = [
        (slot + offset) % slots_per_day
        for offset in range(-tod_neighbour_slots, tod_neighbour_slots + 1)
    ]

    slot_reference[slot] = df.loc[
        df["slot_5min"].isin(neighbour_slots),
        "q_p"
    ].median()

df["q_p_tod_pm30min"] = df["slot_5min"].map(slot_reference)
df["d_q_p_tod_pm30min"] = df["q_p"] - df["q_p_tod_pm30min"]

# BASIC VALIDITY FLAGS
# These are useful later for preprocessing / clustering screening.

feature_cols = [
    "q_p",
    "delta_q_p_5min",
    "delta_q_p_30min",
    "delta_q_p_60min",
    "std_q_p_30min",
    "std_q_p_60min",
    "d_q_p_tod_pm30min",
    "pump_activity",
]

df["has_all_candidate_features"] = df[feature_cols].notna().all(axis=1).astype(int)

# SUMMARY DIAGNOSTICS

summary_records = []

for col in feature_cols:
    s = df[col]

    summary_records.append({
        "feature": col,
        "n_observations": len(s),
        "n_missing": int(s.isna().sum()),
        "missing_share": float(s.isna().mean()),
        "n_unique": int(s.nunique(dropna=True)),
        "min": float(s.min(skipna=True)) if s.notna().any() else np.nan,
        "q25": float(s.quantile(0.25)) if s.notna().any() else np.nan,
        "median": float(s.median(skipna=True)) if s.notna().any() else np.nan,
        "q75": float(s.quantile(0.75)) if s.notna().any() else np.nan,
        "max": float(s.max(skipna=True)) if s.notna().any() else np.nan,
        "iqr": float(s.quantile(0.75) - s.quantile(0.25)) if s.notna().any() else np.nan,
        "zero_share": float((s == 0).mean()) if s.notna().any() else np.nan,
        "most_frequent_value_share": float(s.value_counts(dropna=True, normalize=True).iloc[0]) if s.notna().any() else np.nan,
    })

summary = pd.DataFrame(summary_records)

# Overall summary row
overall_summary = pd.DataFrame([{
    "feature": "__overall__",
    "n_observations": len(df),
    "n_missing": int(df[feature_cols].isna().any(axis=1).sum()),
    "missing_share": float(df[feature_cols].isna().any(axis=1).mean()),
    "n_unique": np.nan,
    "min": np.nan,
    "q25": np.nan,
    "median": np.nan,
    "q75": np.nan,
    "max": np.nan,
    "iqr": np.nan,
    "zero_share": np.nan,
    "most_frequent_value_share": np.nan,
}])

summary = pd.concat([summary, overall_summary], ignore_index=True)


df.to_csv(output_file, sep=sep, index=False)
summary.to_csv(summary_file, sep=sep, index=False)

print("Pump feature construction completed.")
print(f"Input file: {os.path.abspath(input_file)}")
print(f"Summary file: {os.path.abspath(summary_file)}")
print()
print(f"Total timestamps: {len(df)}")
print(f"Timestamps with all candidate features available: {int(df['has_all_candidate_features'].sum())}")
print(f"Pump activity threshold used: {activity_threshold}")
print()
print("Candidate pump features:")
for col in feature_cols:
    print(f" - {col}")

display(df.head(15))
display(summary)


Pump feature construction completed.
Input file: 2019_SCADA_Flows.csv
Output feature file: pump_features_2019/PUMP_1_pump_features_2019.csv
Summary file: pump_features_2019/PUMP_1_pump_features_2019_summary.csv

Total timestamps: 105120
Timestamps with all candidate features available: 105108
Pump activity threshold used: 1e-09

Candidate pump features:
 - q_p
 - delta_q_p_5min
 - delta_q_p_30min
 - delta_q_p_60min
 - std_q_p_30min
 - std_q_p_60min
 - d_q_p_tod_pm30min
 - pump_activity


,Timestamp,q_p,year,month,dayofyear,hour,minute,slot_5min,delta_q_p_5min,delta_q_p_30min,delta_q_p_60min,std_q_p_30min,std_q_p_60min,pump_activity,q_p_tod_pm30min,d_q_p_tod_pm30min,has_all_candidate_features
0,2019-01-01 00:00:00,44.02,2019,1,1,0,0,0,NaN,NaN,NaN,NaN,NaN,1,43.87,0.15,0
1,2019-01-01 00:05:00,44.04,2019,1,1,0,5,1,0.02,NaN,NaN,NaN,NaN,1,43.88,0.16,0
2,2019-01-01 00:10:00,44.04,2019,1,1,0,10,2,0.00,NaN,NaN,NaN,NaN,1,43.89,0.15,0
3,2019-01-01 00:15:00,44.05,2019,1,1,0,15,3,0.01,NaN,NaN,NaN,NaN,1,43.89,0.16,0
4,2019-01-01 00:20:00,44.01,2019,1,1,0,20,4,-0.04,NaN,NaN,NaN,NaN,1,43.90,0.11,0
5,2019-01-01 00:25:00,44.04,2019,1,1,0,25,5,0.03,NaN,NaN,0.015055,NaN,1,43.91,0.13,0
6,2019-01-01 00:30:00,44.06,2019,1,1,0,30,6,0.02,0.04,NaN,0.016733,NaN,1,43.91,0.15,0
7,2019-01-01 00:35:00,44.04,2019,1,1,0,35,7,-0.02,0.00,NaN,0.016733,NaN,1,43.92,0.12,0
8,2019-01-01 00:40:00,44.05,2019,1,1,0,40,8,0.01,0.01,NaN,0.017224,NaN,1,43.92,0.13,0
9,2019-01-01 00:45:00,44.04,2019,1,1,0,45,9,-0.01,-0.01,NaN,0.016733,NaN,1,43.93,0.11,0


,feature,n_observations,n_missing,missing_share,n_unique,min,q25,median,q75,max,iqr,zero_share,most_frequent_value_share
0,q_p,105120,0,0.000000,119.0,0.00,43.460000,43.730000,43.930000,44.380000,0.470000,0.221005,0.221005
1,delta_q_p_5min,105120,1,0.000010,221.0,-44.06,-0.020000,0.000000,0.010000,44.370000,0.030000,0.323031,0.323034
2,delta_q_p_30min,105120,6,0.000057,255.0,-44.07,-0.020000,0.000000,0.020000,44.380000,0.040000,0.280632,0.280648
3,delta_q_p_60min,105120,12,0.000114,283.0,-44.09,-0.030000,0.000000,0.030000,44.380000,0.060000,0.246927,0.246956
4,std_q_p_30min,105120,5,0.000048,74608.0,0.00,0.007528,0.023166,0.036332,24.302450,0.028804,0.211815,0.211825
5,std_q_p_60min,105120,11,0.000105,78493.0,0.00,0.010000,0.029064,0.040527,23.171477,0.030527,0.198212,0.198232
6,d_q_p_tod_pm30min,105120,0,0.000000,275.0,-44.02,-0.220000,0.000000,0.130000,0.720000,0.350000,0.018731,0.018731
7,pump_activity,105120,0,0.000000,2.0,0.00,1.000000,1.000000,1.000000,1.000000,0.000000,0.221005,0.778995
8,__overall__,105120,12,0.000114,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pump-feature diagnostic screening

Screen the 2019 pump-feature matrix using the same diagnostic protocol as in the 2018 main evaluation.

In [3]:
import os
import numpy as np
import pandas as pd
from IPython.display import display


input_file = "pump_features_2019/PUMP_1_pump_features_2019.csv"

output_dir = "pump_preprocessing_screening_2019"
os.makedirs(output_dir, exist_ok=True)

sep = ";"
timestamp_col = "Timestamp"

candidate_features = [
    "q_p",
    "delta_q_p_5min",
    "delta_q_p_30min",
    "delta_q_p_60min",
    "std_q_p_30min",
    "std_q_p_60min",
    "d_q_p_tod_pm30min",
    "pump_activity",
]

# Diagnostic thresholds; these are screening thresholds, not final rules.
missing_share_excessive_threshold = 0.05
dominant_value_share_threshold = 0.995
low_unique_threshold = 1
spearman_redundancy_threshold = 0.90
skew_abs_transform_threshold = 2.0
robust_z_extreme_threshold = 20.0
robust_outlier_multiplier = 10.0

# Expected initial missing values from feature construction
# These are not treated as problematic if they match the lag/window structure.
expected_initial_missing = {
    "q_p": 0,
    "delta_q_p_5min": 1,
    "delta_q_p_30min": 6,
    "delta_q_p_60min": 12,
    "std_q_p_30min": 5,
    "std_q_p_60min": 11,
    "d_q_p_tod_pm30min": 0,
    "pump_activity": 0,
}

# Handles semicolon-separated CSV and decimal comma / decimal point.

def read_semicolon_csv(path, sep=";", timestamp_col="Timestamp"):
    df = pd.read_csv(path, sep=sep, dtype=str)

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="raise")

    for col in df.columns:
        if col == timestamp_col:
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values(timestamp_col).reset_index(drop=True)


df = read_semicolon_csv(input_file, sep=sep, timestamp_col=timestamp_col)

missing_cols = [c for c in candidate_features if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing candidate feature columns: {missing_cols}")


duplicate_mask = df[timestamp_col].duplicated(keep=False)
duplicates = df.loc[duplicate_mask].copy()

duplicate_summary = pd.DataFrame([{
    "n_rows": len(df),
    "n_duplicate_timestamp_rows": int(duplicate_mask.sum()),
    "n_duplicate_timestamps": int(df.loc[duplicate_mask, timestamp_col].nunique()),
    "has_duplicate_timestamps": bool(duplicate_mask.any()),
}])

duplicates_file = os.path.join(output_dir, "duplicate_timestamps_2019.csv")
duplicate_summary_file = os.path.join(output_dir, "duplicate_timestamp_summary_2019.csv")

duplicates.to_csv(duplicates_file, sep=sep, index=False)
duplicate_summary.to_csv(duplicate_summary_file, sep=sep, index=False)


def feature_type(col, s):
    nonmissing = s.dropna()
    unique_vals = set(nonmissing.unique())

    if col == "pump_activity":
        return "binary_indicator"

    if len(unique_vals) <= 2 and unique_vals.issubset({0, 1, 0.0, 1.0}):
        return "binary_indicator"

    if col.startswith("std_"):
        return "nonnegative_variability"

    if col.startswith("delta_") or col.startswith("d_"):
        return "signed_dynamic_or_deviation"

    if col == "q_p":
        return "native_level"

    return "numeric"


def safe_skew(s):
    s = s.dropna()
    if len(s) < 3 or s.nunique() < 2:
        return np.nan
    return float(s.skew())


def safe_quantile(s, q):
    s = s.dropna()
    if len(s) == 0:
        return np.nan
    return float(s.quantile(q))


def most_frequent_value_share(s):
    vc = s.dropna().value_counts(normalize=True)
    if len(vc) == 0:
        return np.nan
    return float(vc.iloc[0])


def robust_extreme_metrics(s):
    x = s.dropna()
    if len(x) == 0:
        return {
            "robust_z_max_abs": np.nan,
            "robust_outlier_share": np.nan,
        }

    med = x.median()
    q25 = x.quantile(0.25)
    q75 = x.quantile(0.75)
    iqr = q75 - q25

    if iqr == 0 or not np.isfinite(iqr):
        return {
            "robust_z_max_abs": np.nan,
            "robust_outlier_share": np.nan,
        }

    robust_z = (x - med).abs() / iqr
    return {
        "robust_z_max_abs": float(robust_z.max()),
        "robust_outlier_share": float((robust_z > robust_outlier_multiplier).mean()),
    }


diagnostics = []

for col in candidate_features:
    s = df[col]
    nonmissing = s.dropna()
    q25 = safe_quantile(s, 0.25)
    q75 = safe_quantile(s, 0.75)
    iqr = q75 - q25 if pd.notna(q25) and pd.notna(q75) else np.nan
    ext = robust_extreme_metrics(s)

    n_missing = int(s.isna().sum())
    expected_missing = expected_initial_missing.get(col, 0)
    unexpected_missing = max(0, n_missing - expected_missing)

    diagnostics.append({
        "feature": col,
        "feature_type": feature_type(col, s),

        "n_total": int(len(s)),
        "n_valid": int(s.notna().sum()),
        "n_missing": n_missing,
        "expected_initial_missing": int(expected_missing),
        "unexpected_missing": int(unexpected_missing),
        "missing_share": float(s.isna().mean()),
        "unexpected_missing_share": float(unexpected_missing / len(s)),

        "first_valid_timestamp": nonmissing.index.min() if len(nonmissing) > 0 else np.nan,
        "n_unique": int(s.nunique(dropna=True)),

        "min": float(s.min(skipna=True)) if s.notna().any() else np.nan,
        "q01": safe_quantile(s, 0.01),
        "q05": safe_quantile(s, 0.05),
        "q25": q25,
        "median": float(s.median(skipna=True)) if s.notna().any() else np.nan,
        "q75": q75,
        "q95": safe_quantile(s, 0.95),
        "q99": safe_quantile(s, 0.99),
        "max": float(s.max(skipna=True)) if s.notna().any() else np.nan,

        "mean": float(s.mean(skipna=True)) if s.notna().any() else np.nan,
        "std": float(s.std(skipna=True)) if s.notna().any() else np.nan,
        "iqr": float(iqr) if pd.notna(iqr) else np.nan,
        "skewness": safe_skew(s),

        "zero_share": float((s == 0).mean()) if s.notna().any() else np.nan,
        "most_frequent_value_share": most_frequent_value_share(s),

        "robust_z_max_abs": ext["robust_z_max_abs"],
        "robust_outlier_share_gt_10iqr": ext["robust_outlier_share"],
    })

diagnostics = pd.DataFrame(diagnostics)

# PREPROCESSING SCREENING

def screen_feature(row):
    feature = row["feature"]
    ftype = row["feature_type"]

    reasons = []

    excessive_missing = row["unexpected_missing_share"] > missing_share_excessive_threshold
    no_valid = row["n_valid"] == 0
    near_constant = (
        row["n_unique"] <= low_unique_threshold
        or (pd.notna(row["iqr"]) and row["iqr"] == 0 and ftype != "binary_indicator")
        or (pd.notna(row["most_frequent_value_share"]) and row["most_frequent_value_share"] >= dominant_value_share_threshold)
    )

    high_skew = pd.notna(row["skewness"]) and abs(row["skewness"]) >= skew_abs_transform_threshold
    extreme_tail = pd.notna(row["robust_z_max_abs"]) and row["robust_z_max_abs"] >= robust_z_extreme_threshold
    outlier_share = pd.notna(row["robust_outlier_share_gt_10iqr"]) and row["robust_outlier_share_gt_10iqr"] > 0

    if no_valid:
        return pd.Series({
            "retain_for_kmeans_screening": False,
            "selected_transformation": "exclude",
            "screening_note": "No valid observations."
        })

    if excessive_missing:
        reasons.append("unexpected missingness exceeds screening threshold")

    if near_constant and ftype != "binary_indicator":
        return pd.Series({
            "retain_for_kmeans_screening": False,
            "selected_transformation": "exclude",
            "screening_note": "Feature is constant or nearly constant."
        })

    if ftype == "binary_indicator":
        if row["n_unique"] < 2:
            return pd.Series({
                "retain_for_kmeans_screening": False,
                "selected_transformation": "exclude",
                "screening_note": "Binary indicator is constant and cannot separate operating states."
            })
        return pd.Series({
            "retain_for_kmeans_screening": True,
            "selected_transformation": "none",
            "screening_note": "Binary activity indicator; do not transform."
        })

    if ftype == "nonnegative_variability":
        if high_skew or extreme_tail or outlier_share:
            reasons.append("nonnegative variability feature with skewness or extreme values")
            return pd.Series({
                "retain_for_kmeans_screening": True,
                "selected_transformation": "log1p",
                "screening_note": "; ".join(reasons)
            })
        return pd.Series({
            "retain_for_kmeans_screening": True,
            "selected_transformation": "none",
            "screening_note": "No transformation required by screening diagnostics."
        })

    if ftype == "signed_dynamic_or_deviation":
        if high_skew or extreme_tail or outlier_share:
            reasons.append("signed feature with skewness or extreme values")
            return pd.Series({
                "retain_for_kmeans_screening": True,
                "selected_transformation": "signed_log1p_abs",
                "screening_note": "; ".join(reasons)
            })
        return pd.Series({
            "retain_for_kmeans_screening": True,
            "selected_transformation": "none",
            "screening_note": "No transformation required by screening diagnostics."
        })

    if ftype == "native_level":
        return pd.Series({
            "retain_for_kmeans_screening": True,
            "selected_transformation": "none",
            "screening_note": "Native pump flow retained in original meaning and later robustly scaled."
        })

    return pd.Series({
        "retain_for_kmeans_screening": True,
        "selected_transformation": "none",
        "screening_note": "Retain unless redundancy screening suggests removal."
    })


screening_decisions = diagnostics.join(
    diagnostics.apply(screen_feature, axis=1)
)


retain_candidates = screening_decisions.loc[
    screening_decisions["retain_for_kmeans_screening"],
    "feature"
].tolist()

spearman_matrix = df[retain_candidates].corr(method="spearman")

redundant_pairs = []

for i, f1 in enumerate(retain_candidates):
    for f2 in retain_candidates[i + 1:]:
        rho = spearman_matrix.loc[f1, f2]

        if pd.isna(rho):
            continue

        if abs(rho) >= spearman_redundancy_threshold:
            redundant_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "spearman_rho": float(rho),
                "abs_spearman_rho": float(abs(rho)),
            })

redundant_pairs = pd.DataFrame(redundant_pairs)

# Add interpretation to redundant pairs
def redundancy_note(row):
    f1 = row["feature_1"]
    f2 = row["feature_2"]

    pair = {f1, f2}

    if pair == {"q_p", "pump_activity"}:
        return (
            "Both features were retained for screening: q_p preserves continuous pump-flow level and "
            "pump_activity provides an interpretable inactive/active indicator."
        )

    if f1.startswith("std_") and f2.startswith("std_"):
        return (
            "Rolling variability features are strongly redundant; consider retaining the window "
            "that gives better operational interpretability in k-means screening."
        )

    if f1.startswith("delta_") and f2.startswith("delta_"):
        return (
            "Flow-change features are strongly redundant; consider whether the shorter window "
            "captures switching and the longer window captures broader transitions."
        )

    if "d_q_p_tod_pm30min" in pair and "q_p" in pair:
        return (
            "Native level and time-of-day deviation are strongly associated; retain both only if "
            "daily-profile deviation adds interpretable state separation beyond pump-flow level."
        )

    return (
        "Strong monotonic redundancy; final selection uses interpretability and methodological relevance."
    )

if len(redundant_pairs) > 0:
    redundant_pairs["screening_note"] = redundant_pairs.apply(redundancy_note, axis=1)
else:
    redundant_pairs = pd.DataFrame(columns=[
        "feature_1",
        "feature_2",
        "spearman_rho",
        "abs_spearman_rho",
        "screening_note",
    ])


complete_all_candidate_features = df[candidate_features].notna().all(axis=1)
complete_retained_features = df[retain_candidates].notna().all(axis=1) if retain_candidates else pd.Series(False, index=df.index)

usable_summary = pd.DataFrame([{
    "n_total_rows": len(df),
    "n_duplicate_timestamp_rows": int(duplicate_mask.sum()),
    "n_rows_complete_all_candidate_features": int(complete_all_candidate_features.sum()),
    "share_complete_all_candidate_features": float(complete_all_candidate_features.mean()),
    "n_rows_complete_retained_screening_features": int(complete_retained_features.sum()),
    "share_complete_retained_screening_features": float(complete_retained_features.mean()),
    "first_timestamp": df[timestamp_col].min(),
    "last_timestamp": df[timestamp_col].max(),
    "n_candidate_features": len(candidate_features),
    "n_retained_after_basic_screening": len(retain_candidates),
}])


diagnostics_file = os.path.join(output_dir, "pump_feature_diagnostics_2019.csv")
screening_decisions_file = os.path.join(output_dir, "pump_feature_preprocessing_screening_decisions_2019.csv")
spearman_file = os.path.join(output_dir, "pump_feature_spearman_matrix_2019.csv")
redundant_pairs_file = os.path.join(output_dir, "pump_feature_spearman_redundant_pairs_abs_ge_0_90_2019.csv")
usable_summary_file = os.path.join(output_dir, "pump_feature_usable_rows_summary_2019.csv")

diagnostics.to_csv(diagnostics_file, sep=sep, index=False)
screening_decisions.to_csv(screening_decisions_file, sep=sep, index=False)
spearman_matrix.to_csv(spearman_file, sep=sep)
redundant_pairs.to_csv(redundant_pairs_file, sep=sep, index=False)
usable_summary.to_csv(usable_summary_file, sep=sep, index=False)


print("Pump feature preprocessing screening completed.")
print()
print()

print("Usable rows summary:")
display(usable_summary)

print("Feature diagnostics:")
display(diagnostics)

print("Preprocessing screening decisions:")
display(screening_decisions[[
    "feature",
    "feature_type",
    "n_missing",
    "expected_initial_missing",
    "unexpected_missing",
    "n_unique",
    "iqr",
    "skewness",
    "zero_share",
    "most_frequent_value_share",
    "robust_z_max_abs",
    "retain_for_kmeans_screening",
    "selected_transformation",
    "screening_note",
]])

print("Spearman redundant pairs:")
display(redundant_pairs.sort_values("abs_spearman_rho", ascending=False) if len(redundant_pairs) else redundant_pairs)


Pump feature preprocessing screening completed.

Usable rows summary:


,n_total_rows,n_duplicate_timestamp_rows,n_rows_complete_all_candidate_features,share_complete_all_candidate_features,n_rows_complete_retained_screening_features,share_complete_retained_screening_features,first_timestamp,last_timestamp,n_candidate_features,n_retained_after_basic_screening
0,105120,0,105108,0.999886,105108,0.999886,2019-01-01,2019-12-31 23:55:00,8,8


Feature diagnostics:


,feature,feature_type,n_total,n_valid,n_missing,expected_initial_missing,unexpected_missing,missing_share,unexpected_missing_share,first_valid_timestamp,...,q99,max,mean,std,iqr,skewness,zero_share,most_frequent_value_share,robust_z_max_abs,robust_outlier_share_gt_10iqr
0,q_p,native_level,105120,105120,0,0,0,0.000000,0.0,0,...,44.200000,44.380000,34.129913,18.179887,0.470000,-1.344478,0.221005,0.221005,93.042553,0.221005
1,delta_q_p_5min,signed_dynamic_or_deviation,105120,105119,1,1,0,0.000010,0.0,1,...,0.110000,44.370000,-0.000003,2.826541,0.030000,0.086360,0.323031,0.323034,1479.000000,0.004148
2,delta_q_p_30min,signed_dynamic_or_deviation,105120,105114,6,6,0,0.000057,0.0,6,...,43.770000,44.380000,-0.000020,6.923297,0.040000,0.033848,0.280632,0.280648,1109.500000,0.024887
3,delta_q_p_60min,signed_dynamic_or_deviation,105120,105108,12,12,0,0.000114,0.0,12,...,44.030000,44.380000,-0.000044,9.791347,0.060000,0.023073,0.246927,0.246956,739.666667,0.049777
4,std_q_p_30min,nonnegative_variability,105120,105115,5,5,0,0.000048,0.0,5,...,22.602556,24.302450,0.458518,3.018267,0.028804,6.882085,0.211815,0.211825,842.911335,0.020739
5,std_q_p_60min,nonnegative_variability,105120,105109,11,11,0,0.000105,0.0,11,...,22.539475,23.171477,0.897934,4.062444,0.030527,4.583157,0.198212,0.198232,758.100684,0.045629
6,d_q_p_tod_pm30min,signed_dynamic_or_deviation,105120,105120,0,0,0,0.000000,0.0,0,...,0.420000,0.720000,-9.615330,18.164440,0.350000,-1.344690,0.018731,0.018731,125.771429,0.221005
7,pump_activity,binary_indicator,105120,105120,0,0,0,0.000000,0.0,0,...,1.000000,1.000000,0.778995,0.414926,0.000000,-1.344823,0.221005,0.778995,NaN,NaN


Preprocessing screening decisions:


,feature,feature_type,n_missing,expected_initial_missing,unexpected_missing,n_unique,iqr,skewness,zero_share,most_frequent_value_share,robust_z_max_abs,retain_for_kmeans_screening,recommended_transformation,screening_note
0,q_p,native_level,0,0,0,119,0.470000,-1.344478,0.221005,0.221005,93.042553,True,none,Native pump flow retained in original meaning ...
1,delta_q_p_5min,signed_dynamic_or_deviation,1,1,0,221,0.030000,0.086360,0.323031,0.323034,1479.000000,True,signed_log1p_abs,signed feature with skewness or extreme values
2,delta_q_p_30min,signed_dynamic_or_deviation,6,6,0,255,0.040000,0.033848,0.280632,0.280648,1109.500000,True,signed_log1p_abs,signed feature with skewness or extreme values
3,delta_q_p_60min,signed_dynamic_or_deviation,12,12,0,283,0.060000,0.023073,0.246927,0.246956,739.666667,True,signed_log1p_abs,signed feature with skewness or extreme values
4,std_q_p_30min,nonnegative_variability,5,5,0,74453,0.028804,6.882085,0.211815,0.211825,842.911335,True,log1p,nonnegative variability feature with skewness ...
5,std_q_p_60min,nonnegative_variability,11,11,0,78396,0.030527,4.583157,0.198212,0.198232,758.100684,True,log1p,nonnegative variability feature with skewness ...
6,d_q_p_tod_pm30min,signed_dynamic_or_deviation,0,0,0,275,0.350000,-1.344690,0.018731,0.018731,125.771429,True,signed_log1p_abs,signed feature with skewness or extreme values
7,pump_activity,binary_indicator,0,0,0,2,0.000000,-1.344823,0.221005,0.778995,NaN,True,none,Binary activity indicator; do not transform.


Spearman redundant pairs:


,feature_1,feature_2,spearman_rho,abs_spearman_rho,screening_note


## Pump-feature transformation and robust scaling

Apply the retained transformations and robust scaling before k-means state extraction.

In [4]:
import os
import numpy as np
import pandas as pd
from IPython.display import display


input_file = "pump_features_2019/PUMP_1_pump_features_2019.csv"

output_dir = "pump_kmeans_inputs_2019"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "PUMP_1_kmeans_input_2019.csv")
transformation_file = os.path.join(output_dir, "PUMP_1_applied_transformations_2019.csv")
scaling_file = os.path.join(output_dir, "PUMP_1_robust_scaling_parameters_2019.csv")
summary_file = os.path.join(output_dir, "PUMP_1_kmeans_input_summary_2019.csv")

sep = ";"
timestamp_col = "Timestamp"

# Candidate features retained after screening
native_features = [
    "q_p",
    "pump_activity",
]

signed_log_features = [
    "delta_q_p_5min",
    "delta_q_p_30min",
    "delta_q_p_60min",
    "d_q_p_tod_pm30min",
]

log1p_features = [
    "std_q_p_30min",
    "std_q_p_60min",
]

all_required_features = native_features + signed_log_features + log1p_features

# Handles semicolon-separated CSV and decimal comma / decimal point.

def read_semicolon_csv(path, sep=";", timestamp_col="Timestamp"):
    df = pd.read_csv(path, sep=sep, dtype=str)

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="raise")

    for col in df.columns:
        if col == timestamp_col:
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values(timestamp_col).reset_index(drop=True)


def signed_log1p_abs(x):
    return np.sign(x) * np.log1p(np.abs(x))


def robust_scale(series):
    median = series.median()
    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    iqr = q75 - q25

    if pd.isna(iqr) or iqr == 0:
        scaled = series - median
    else:
        scaled = (series - median) / iqr

    return scaled, median, q25, q75, iqr


# LOAD DATA

df = read_semicolon_csv(input_file, sep=sep, timestamp_col=timestamp_col)

missing_cols = [c for c in all_required_features if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required feature columns: {missing_cols}")

# APPLY TRANSFORMATIONS

out = df[[timestamp_col] + all_required_features].copy()

transformation_records = []

# Native features retained without transformation
for col in native_features:
    new_col = f"{col}_transformed"
    out[new_col] = out[col]

    transformation_records.append({
        "original_feature": col,
        "transformed_feature": new_col,
        "transformation": "none",
        "reason": "Native pump flow or binary activity indicator retained in original meaning."
    })

# Signed log transform for signed dynamic/deviation features
for col in signed_log_features:
    new_col = f"{col}_signed_log1p_abs"
    out[new_col] = signed_log1p_abs(out[col])

    transformation_records.append({
        "original_feature": col,
        "transformed_feature": new_col,
        "transformation": "sign(x) * log(1 + abs(x))",
        "reason": "Signed change/deviation feature with extreme values; direction is preserved while large magnitudes are compressed."
    })

# log1p transform for non-negative rolling variability features
for col in log1p_features:
    if (out[col].dropna() < 0).any():
        raise ValueError(f"Feature {col} contains negative values, but log1p was requested.")

    new_col = f"{col}_log1p"
    out[new_col] = np.log1p(out[col])

    transformation_records.append({
        "original_feature": col,
        "transformed_feature": new_col,
        "transformation": "log(1 + x)",
        "reason": "Non-negative rolling variability feature with skewness/extreme values."
    })

transformations = pd.DataFrame(transformation_records)

transformed_features = transformations["transformed_feature"].tolist()

# REMOVE ROWS WITH INCOMPLETE TRANSFORMED FEATURES

out["complete_kmeans_features"] = out[transformed_features].notna().all(axis=1).astype(int)

n_before = len(out)
out_complete = out.loc[out["complete_kmeans_features"] == 1].copy()
n_after = len(out_complete)

# ROBUST SCALING

scaling_records = []
scaled_features = []

for col in transformed_features:
    scaled_col = f"{col}_scaled"
    out_complete[scaled_col], median, q25, q75, iqr = robust_scale(out_complete[col])

    scaled_features.append(scaled_col)

    scaling_records.append({
        "transformed_feature": col,
        "scaled_feature": scaled_col,
        "median": median,
        "q25": q25,
        "q75": q75,
        "iqr": iqr,
        "scaling": "robust scaling: (x - median) / IQR",
        "used_fallback_center_only": bool(pd.isna(iqr) or iqr == 0),
    })

scaling_params = pd.DataFrame(scaling_records)

# ADD METADATA FOR K-MEANS

out_complete["kmeans_input_row"] = np.arange(len(out_complete))

# This column is useful later when selecting features programmatically
feature_columns_table = pd.DataFrame({
    "kmeans_feature_column": scaled_features
})

feature_columns_file = os.path.join(output_dir, "PUMP_1_kmeans_feature_columns_2019.csv")
feature_columns_table.to_csv(feature_columns_file, sep=sep, index=False)

# SUMMARY

summary = pd.DataFrame([{
    "input_file": input_file,
    "output_file": output_file,
    "n_rows_before_complete_case_filter": n_before,
    "n_rows_after_complete_case_filter": n_after,
    "n_removed_rows": n_before - n_after,
    "share_retained": n_after / n_before if n_before > 0 else np.nan,
    "n_original_required_features": len(all_required_features),
    "n_transformed_features": len(transformed_features),
    "n_scaled_kmeans_features": len(scaled_features),
    "first_timestamp_retained": out_complete[timestamp_col].min(),
    "last_timestamp_retained": out_complete[timestamp_col].max(),
}])


out_complete.to_csv(output_file, sep=sep, index=False)
transformations.to_csv(transformation_file, sep=sep, index=False)
scaling_params.to_csv(scaling_file, sep=sep, index=False)
summary.to_csv(summary_file, sep=sep, index=False)

print("Pump feature preprocessing completed.")
print()
print()
print("Scaled k-means feature columns:")
for col in scaled_features:
    print(f" - {col}")

print()
print("Summary:")
display(summary)

print("Applied transformations:")
display(transformations)

print("Robust scaling parameters:")
display(scaling_params)

print("Preview of k-means input:")
display(out_complete[[timestamp_col] + scaled_features].head(10))


Pump feature preprocessing completed.

Scaled k-means feature columns:
 - q_p_transformed_scaled
 - pump_activity_transformed_scaled
 - delta_q_p_5min_signed_log1p_abs_scaled
 - delta_q_p_30min_signed_log1p_abs_scaled
 - delta_q_p_60min_signed_log1p_abs_scaled
 - d_q_p_tod_pm30min_signed_log1p_abs_scaled
 - std_q_p_30min_log1p_scaled
 - std_q_p_60min_log1p_scaled

Summary:


,input_file,output_file,n_rows_before_complete_case_filter,n_rows_after_complete_case_filter,n_removed_rows,share_retained,n_original_required_features,n_transformed_features,n_scaled_kmeans_features,first_timestamp_retained,last_timestamp_retained
0,pump_features_2019/PUMP_1_pump_features_2019.csv,pump_kmeans_inputs_2019\PUMP_1_kmeans_input_20...,105120,105108,12,0.999886,8,8,8,2019-01-01 01:00:00,2019-12-31 23:55:00


Applied transformations:


,original_feature,transformed_feature,transformation,reason
0,q_p,q_p_transformed,none,Native pump flow or binary activity indicator ...
1,pump_activity,pump_activity_transformed,none,Native pump flow or binary activity indicator ...
2,delta_q_p_5min,delta_q_p_5min_signed_log1p_abs,sign(x) * log(1 + abs(x)),Signed change/deviation feature with extreme v...
3,delta_q_p_30min,delta_q_p_30min_signed_log1p_abs,sign(x) * log(1 + abs(x)),Signed change/deviation feature with extreme v...
4,delta_q_p_60min,delta_q_p_60min_signed_log1p_abs,sign(x) * log(1 + abs(x)),Signed change/deviation feature with extreme v...
5,d_q_p_tod_pm30min,d_q_p_tod_pm30min_signed_log1p_abs,sign(x) * log(1 + abs(x)),Signed change/deviation feature with extreme v...
6,std_q_p_30min,std_q_p_30min_log1p,log(1 + x),Non-negative rolling variability feature with ...
7,std_q_p_60min,std_q_p_60min_log1p,log(1 + x),Non-negative rolling variability feature with ...


Robust scaling parameters:


,transformed_feature,scaled_feature,median,q25,q75,iqr,scaling,used_fallback_center_only
0,q_p_transformed,q_p_transformed_scaled,43.730000,43.460000,43.930000,0.470000,robust scaling: (x - median) / IQR,False
1,pump_activity_transformed,pump_activity_transformed_scaled,1.000000,1.000000,1.000000,0.000000,robust scaling: (x - median) / IQR,True
2,delta_q_p_5min_signed_log1p_abs,delta_q_p_5min_signed_log1p_abs_scaled,0.000000,-0.019803,0.009950,0.029753,robust scaling: (x - median) / IQR,False
3,delta_q_p_30min_signed_log1p_abs,delta_q_p_30min_signed_log1p_abs_scaled,0.000000,-0.019803,0.019803,0.039605,robust scaling: (x - median) / IQR,False
4,delta_q_p_60min_signed_log1p_abs,delta_q_p_60min_signed_log1p_abs_scaled,0.000000,-0.029559,0.029559,0.059118,robust scaling: (x - median) / IQR,False
5,d_q_p_tod_pm30min_signed_log1p_abs,d_q_p_tod_pm30min_signed_log1p_abs_scaled,0.000000,-0.198851,0.122218,0.321068,robust scaling: (x - median) / IQR,False
6,std_q_p_30min_log1p,std_q_p_30min_log1p_scaled,0.022902,0.007500,0.035687,0.028188,robust scaling: (x - median) / IQR,False
7,std_q_p_60min_log1p,std_q_p_60min_log1p_scaled,0.028649,0.009950,0.039727,0.029777,robust scaling: (x - median) / IQR,False


Preview of k-means input:


,Timestamp,q_p_transformed_scaled,pump_activity_transformed_scaled,delta_q_p_5min_signed_log1p_abs_scaled,delta_q_p_30min_signed_log1p_abs_scaled,delta_q_p_60min_signed_log1p_abs_scaled,d_q_p_tod_pm30min_signed_log1p_abs_scaled,std_q_p_30min_log1p_scaled,std_q_p_60min_log1p_scaled
12,2019-01-01 01:00:00,0.680851,0.0,-0.334432,-0.251238,0.500000,0.296853,-0.496572,-0.501294
13,2019-01-01 01:05:00,0.702128,0.0,0.334432,0.500000,0.334970,0.325040,-0.523985,-0.483483
14,2019-01-01 01:10:00,0.680851,0.0,-0.334432,0.000000,0.168314,0.268409,-0.523985,-0.487871
15,2019-01-01 01:15:00,0.702128,0.0,0.334432,0.500000,0.168314,0.296853,-0.629744,-0.473114
16,2019-01-01 01:20:00,0.702128,0.0,0.000000,0.000000,0.825307,0.268409,-0.629744,-0.672549
17,2019-01-01 01:25:00,0.723404,0.0,0.334432,0.251238,0.500000,0.296853,-0.546416,-0.659730
18,2019-01-01 01:30:00,0.723404,0.0,0.000000,0.500000,0.168314,0.268409,-0.546416,-0.629234
19,2019-01-01 01:35:00,0.702128,0.0,-0.334432,0.000000,0.334970,0.239703,-0.546416,-0.672549
20,2019-01-01 01:40:00,0.723404,0.0,0.334432,0.500000,0.334970,0.268409,-0.618690,-0.661127
21,2019-01-01 01:45:00,0.723404,0.0,0.000000,0.251238,0.500000,0.239703,-0.629744,-0.721956


## Pump-state screening

Evaluate candidate k-means state resolutions for the 2019 pump-flow feature space.

In [5]:
import os
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score


input_file = "pump_kmeans_inputs_2019/PUMP_1_kmeans_input_2019.csv"
feature_columns_file = "pump_kmeans_inputs_2019/PUMP_1_kmeans_feature_columns_2019.csv"

output_dir = "pump_kmeans_screening_2019"
os.makedirs(output_dir, exist_ok=True)

sep = ";"
timestamp_col = "Timestamp"

k_min = 2
k_max = 12
random_state = 42
n_init = 50

# SCADA sampling interval
sampling_minutes = 5
short_episode_threshold_minutes = 30

# Silhouette can be expensive on 105k rows, so use a fixed sample.
silhouette_sample_size = 20000

metrics_file = os.path.join(output_dir, "pump_kmeans_screening_metrics_k2_12.csv")
state_sizes_file = os.path.join(output_dir, "pump_kmeans_state_sizes_k2_12.csv")
episode_summary_file = os.path.join(output_dir, "pump_kmeans_episode_summaries_k2_12.csv")
state_episode_summary_file = os.path.join(output_dir, "pump_kmeans_state_episode_summaries_k2_12.csv")
state_profiles_file = os.path.join(output_dir, "pump_kmeans_state_profiles_k2_12.csv")
labels_file = os.path.join(output_dir, "pump_kmeans_labels_k2_12.csv")

# ROBUST CSV LOADING
# Handles semicolon-separated CSV and decimal comma / decimal point.

def read_semicolon_csv(path, sep=";", timestamp_col="Timestamp"):
    df = pd.read_csv(path, sep=sep, dtype=str)

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="raise")

    for col in df.columns:
        if col == timestamp_col:
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values(timestamp_col).reset_index(drop=True)


# LOAD DATA

df = read_semicolon_csv(input_file, sep=sep, timestamp_col=timestamp_col)

feature_cols_df = pd.read_csv(feature_columns_file, sep=sep)
if "kmeans_feature_column" not in feature_cols_df.columns:
    raise ValueError("Feature column file must contain 'kmeans_feature_column'.")

kmeans_features = feature_cols_df["kmeans_feature_column"].tolist()

missing_features = [c for c in kmeans_features if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing k-means feature columns in input file: {missing_features}")

# Complete-case input for clustering
complete_mask = df[kmeans_features].notna().all(axis=1)
df_k = df.loc[complete_mask].copy().reset_index(drop=True)

X = df_k[kmeans_features].to_numpy(dtype=float)

print(f"Rows in input file: {len(df)}")
print(f"Rows used for k-means: {len(df_k)}")
print(f"K-means features: {len(kmeans_features)}")
for col in kmeans_features:
    print(f" - {col}")

# HELPERS

def build_episode_table(labels, timestamps, k):
    tmp = pd.DataFrame({
        "Timestamp": timestamps,
        "state": labels
    }).sort_values("Timestamp").reset_index(drop=True)

    tmp["state_previous"] = tmp["state"].shift(1)
    tmp["new_episode"] = (tmp["state"] != tmp["state_previous"]).astype(int)
    tmp.loc[0, "new_episode"] = 1
    tmp["episode_id"] = tmp["new_episode"].cumsum()

    ep = (
        tmp.groupby("episode_id")
        .agg(
            k=("state", lambda x: k),
            state=("state", "first"),
            start_time=("Timestamp", "min"),
            end_time=("Timestamp", "max"),
            n_timestamps=("Timestamp", "size"),
        )
        .reset_index()
    )

    ep["duration_minutes"] = ep["n_timestamps"] * sampling_minutes
    ep["is_short_episode"] = ep["duration_minutes"] < short_episode_threshold_minutes

    return tmp[["Timestamp", "state", "episode_id"]], ep


def summarize_episodes(ep, k):
    d = ep["duration_minutes"]

    return {
        "k": k,
        "n_episodes": int(len(ep)),
        "min_episode_duration_min": float(d.min()),
        "q25_episode_duration_min": float(d.quantile(0.25)),
        "median_episode_duration_min": float(d.median()),
        "mean_episode_duration_min": float(d.mean()),
        "q75_episode_duration_min": float(d.quantile(0.75)),
        "max_episode_duration_min": float(d.max()),
        "n_short_episodes_lt_30min": int(ep["is_short_episode"].sum()),
        "share_short_episodes_lt_30min": float(ep["is_short_episode"].mean()),
    }


def summarize_state_episodes(ep, k):
    rows = []

    for state, g in ep.groupby("state"):
        d = g["duration_minutes"]

        rows.append({
            "k": k,
            "state": int(state),
            "n_episodes": int(len(g)),
            "min_episode_duration_min": float(d.min()),
            "q25_episode_duration_min": float(d.quantile(0.25)),
            "median_episode_duration_min": float(d.median()),
            "mean_episode_duration_min": float(d.mean()),
            "q75_episode_duration_min": float(d.quantile(0.75)),
            "max_episode_duration_min": float(d.max()),
            "n_short_episodes_lt_30min": int(g["is_short_episode"].sum()),
            "share_short_episodes_lt_30min": float(g["is_short_episode"].mean()),
        })

    return rows


def make_state_profiles(df_with_labels, k, label_col):
    # Prefer original interpretable variables where available.
    interpretable_cols = [
        "q_p",
        "delta_q_p_5min",
        "delta_q_p_30min",
        "delta_q_p_60min",
        "std_q_p_30min",
        "std_q_p_60min",
        "d_q_p_tod_pm30min",
        "pump_activity",
    ]

    profile_cols = [c for c in interpretable_cols if c in df_with_labels.columns]

    rows = []

    for state, g in df_with_labels.groupby(label_col):
        row = {
            "k": k,
            "state": int(state),
            "n_timestamps": int(len(g)),
            "share_timestamps": float(len(g) / len(df_with_labels)),
        }

        for col in profile_cols:
            row[f"{col}_median"] = float(g[col].median(skipna=True))
            row[f"{col}_q25"] = float(g[col].quantile(0.25))
            row[f"{col}_q75"] = float(g[col].quantile(0.75))
            row[f"{col}_mean"] = float(g[col].mean(skipna=True))

        rows.append(row)

    return rows


# K-MEANS SCREENING

all_metrics = []
all_state_sizes = []
all_episode_summaries = []
all_state_episode_summaries = []
all_state_profiles = []

labels_out = df_k[[timestamp_col]].copy()

for k in range(k_min, k_max + 1):
    print(f"Running k-means for k={k}...")

    km = KMeans(
        n_clusters=k,
        n_init=n_init,
        random_state=random_state,
        algorithm="lloyd"
    )

    labels = km.fit_predict(X)
    label_col = f"pump_state_k{k}"
    labels_out[label_col] = labels

    # ---------- Internal metrics ----------
    inertia = float(km.inertia_)

    if len(X) > silhouette_sample_size:
        rng = np.random.default_rng(random_state)
        sample_idx = rng.choice(len(X), size=silhouette_sample_size, replace=False)
        silhouette = float(
            silhouette_score(
                X[sample_idx],
                labels[sample_idx],
                metric="euclidean"
            )
        )
        silhouette_n = silhouette_sample_size
    else:
        silhouette = float(silhouette_score(X, labels, metric="euclidean"))
        silhouette_n = len(X)

    calinski = float(calinski_harabasz_score(X, labels))
    davies = float(davies_bouldin_score(X, labels))

    state_counts = pd.Series(labels).value_counts().sort_index()
    min_state_count = int(state_counts.min())
    max_state_count = int(state_counts.max())
    min_state_share = float(state_counts.min() / len(labels))
    max_state_share = float(state_counts.max() / len(labels))

    # ---------- Episodes ----------
    timestamp_state_table, ep = build_episode_table(
        labels=labels,
        timestamps=df_k[timestamp_col],
        k=k
    )

    ep_summary = summarize_episodes(ep, k)
    all_episode_summaries.append(ep_summary)
    all_state_episode_summaries.extend(summarize_state_episodes(ep, k))

    # ---------- Metrics row ----------
    all_metrics.append({
        "k": k,
        "n_rows": int(len(X)),
        "n_features": int(X.shape[1]),
        "inertia": inertia,
        "silhouette": silhouette,
        "silhouette_sample_size": int(silhouette_n),
        "calinski_harabasz": calinski,
        "davies_bouldin": davies,
        "min_state_count": min_state_count,
        "max_state_count": max_state_count,
        "min_state_share": min_state_share,
        "max_state_share": max_state_share,
        "n_episodes": ep_summary["n_episodes"],
        "median_episode_duration_min": ep_summary["median_episode_duration_min"],
        "share_short_episodes_lt_30min": ep_summary["share_short_episodes_lt_30min"],
    })

    # ---------- State sizes ----------
    for state, count in state_counts.items():
        all_state_sizes.append({
            "k": k,
            "state": int(state),
            "n_timestamps": int(count),
            "share_timestamps": float(count / len(labels)),
        })

    # ---------- State profiles ----------
    df_profile = df_k.copy()
    df_profile[label_col] = labels
    all_state_profiles.extend(make_state_profiles(df_profile, k, label_col))


metrics = pd.DataFrame(all_metrics)
state_sizes = pd.DataFrame(all_state_sizes)
episode_summaries = pd.DataFrame(all_episode_summaries)
state_episode_summaries = pd.DataFrame(all_state_episode_summaries)
state_profiles = pd.DataFrame(all_state_profiles)

metrics.to_csv(metrics_file, sep=sep, index=False)
state_sizes.to_csv(state_sizes_file, sep=sep, index=False)
episode_summaries.to_csv(episode_summary_file, sep=sep, index=False)
state_episode_summaries.to_csv(state_episode_summary_file, sep=sep, index=False)
state_profiles.to_csv(state_profiles_file, sep=sep, index=False)
labels_out.to_csv(labels_file, sep=sep, index=False)

print("\nK-means screening completed.")

print("\nScreening metrics:")
display(metrics)

print("\nState sizes:")
display(state_sizes.head(50))

print("\nEpisode summaries:")
display(episode_summaries)

print("\nState profiles:")
display(state_profiles.head(50))


Rows in input file: 105108
Rows used for k-means: 105108
K-means features: 8
 - q_p_transformed_scaled
 - pump_activity_transformed_scaled
 - delta_q_p_5min_signed_log1p_abs_scaled
 - delta_q_p_30min_signed_log1p_abs_scaled
 - delta_q_p_60min_signed_log1p_abs_scaled
 - d_q_p_tod_pm30min_signed_log1p_abs_scaled
 - std_q_p_30min_log1p_scaled
 - std_q_p_60min_log1p_scaled
Running k-means for k=2...
Running k-means for k=3...
Running k-means for k=4...
Running k-means for k=5...
Running k-means for k=6...
Running k-means for k=7...
Running k-means for k=8...
Running k-means for k=9...
Running k-means for k=10...
Running k-means for k=11...
Running k-means for k=12...

K-means screening completed.

Screening metrics:


,k,n_rows,n_features,inertia,silhouette,silhouette_sample_size,calinski_harabasz,davies_bouldin,min_state_count,max_state_count,min_state_share,max_state_share,n_episodes,median_episode_duration_min,share_short_episodes_lt_30min
0,2,105108,8,1.207901e+08,0.867476,20000,1.419949e+05,0.397037,23232,81876,0.221030,0.778970,437,725.0,0.000000
1,3,105108,8,7.588395e+07,0.906274,20000,1.441099e+05,0.628039,2398,79478,0.022815,0.756156,655,520.0,0.000000
2,4,105108,8,3.409052e+07,0.941561,20000,2.568036e+05,0.704992,2398,79478,0.022815,0.756156,873,160.0,0.000000
3,5,105108,8,2.277951e+07,0.948789,20000,3.012826e+05,0.555058,1090,79478,0.010370,0.756156,1091,55.0,0.199817
4,6,105108,8,1.157109e+07,0.956739,20000,4.948550e+05,0.419004,1090,79478,0.010370,0.756156,1309,30.0,0.333079
5,7,105108,8,8.670919e+06,0.959340,20000,5.561614e+05,0.319969,218,79478,0.002074,0.756156,1527,30.0,0.428291
6,8,105108,8,5.777777e+06,0.962269,20000,7.229271e+05,0.245801,218,79478,0.002074,0.756156,1745,30.0,0.499713
7,9,105108,8,4.087328e+06,0.965108,20000,8.996024e+05,0.187820,218,79478,0.002074,0.756156,1963,20.0,0.666327
8,10,105108,8,2.403979e+06,0.967461,20000,1.367751e+06,0.065089,218,79478,0.002074,0.756156,2181,20.0,0.799633
9,11,105108,8,1.500884e+06,0.959069,20000,1.977970e+06,0.066672,218,79260,0.002074,0.754082,2399,20.0,0.817841



State sizes:


,k,state,n_timestamps,share_timestamps
0,2,0,23232,0.221030
1,2,1,81876,0.778970
2,3,0,79478,0.756156
3,3,1,23232,0.221030
4,3,2,2398,0.022815
5,4,0,79478,0.756156
6,4,1,2398,0.022815
7,4,2,20834,0.198215
8,4,3,2398,0.022815
9,5,0,2398,0.022815



Episode summaries:


,k,n_episodes,min_episode_duration_min,q25_episode_duration_min,median_episode_duration_min,mean_episode_duration_min,q75_episode_duration_min,max_episode_duration_min,n_short_episodes_lt_30min,share_short_episodes_lt_30min
0,2,437,160.0,520.0,725.0,1202.608696,1470.0,7070.0,0,0.000000
1,3,655,55.0,55.0,520.0,802.351145,1025.0,7015.0,0,0.000000
2,4,873,55.0,55.0,160.0,601.993127,670.0,7015.0,0,0.000000
3,5,1091,25.0,30.0,55.0,481.704858,540.0,7015.0,218,0.199817
4,6,1309,25.0,25.0,30.0,401.482047,465.0,7015.0,436,0.333079
5,7,1527,5.0,20.0,30.0,344.165029,415.0,7015.0,654,0.428291
6,8,1745,5.0,20.0,30.0,301.169054,160.0,7015.0,872,0.499713
7,9,1963,5.0,5.0,20.0,267.722873,30.0,7015.0,1308,0.666327
8,10,2181,5.0,5.0,20.0,240.962861,25.0,7015.0,1744,0.799633
9,11,2399,5.0,5.0,20.0,219.066278,25.0,7010.0,1962,0.817841



State profiles:


,k,state,n_timestamps,share_timestamps,q_p_median,q_p_q25,q_p_q75,q_p_mean,delta_q_p_5min_median,delta_q_p_5min_q25,...,std_q_p_60min_q75,std_q_p_60min_mean,d_q_p_tod_pm30min_median,d_q_p_tod_pm30min_q25,d_q_p_tod_pm30min_q75,d_q_p_tod_pm30min_mean,pump_activity_median,pump_activity_q25,pump_activity_q75,pump_activity_mean
0,2,0,23232,0.221030,0.00,0.0000,0.0000,0.000000,0.00,0.00,...,0.000000,1.971930,-43.720,-43.8900,-43.5400,-43.716471,0.0,0.0,0.0,0.0
1,2,1,81876,0.778970,43.82,43.6600,43.9800,43.812692,0.00,-0.02,...,0.042310,0.593203,0.060,-0.0500,0.1600,0.059309,1.0,1.0,1.0,1.0
2,3,0,79478,0.756156,43.81,43.6500,43.9700,43.808238,0.00,-0.02,...,0.041341,0.032635,0.050,-0.0500,0.1500,0.052497,1.0,1.0,1.0,1.0
3,3,1,23232,0.221030,0.00,0.0000,0.0000,0.000000,0.00,0.00,...,0.000000,1.971930,-43.720,-43.8900,-43.5400,-43.716471,0.0,0.0,0.0,0.0
4,3,2,2398,0.022815,43.99,43.8000,44.0900,43.960313,0.00,-0.02,...,22.514068,19.172360,0.280,0.2000,0.3700,0.285075,1.0,1.0,1.0,1.0
5,4,0,79478,0.756156,43.81,43.6500,43.9700,43.808238,0.00,-0.02,...,0.041341,0.032635,0.050,-0.0500,0.1500,0.052497,1.0,1.0,1.0,1.0
6,4,1,2398,0.022815,0.00,0.0000,0.0000,0.000000,0.00,0.00,...,22.419276,19.104200,-43.890,-43.9900,-43.7200,-43.843920,0.0,0.0,0.0,0.0
7,4,2,20834,0.198215,0.00,0.0000,0.0000,0.000000,0.00,0.00,...,0.000000,0.000000,-43.680,-43.8600,-43.5300,-43.701801,0.0,0.0,0.0,0.0
8,4,3,2398,0.022815,43.99,43.8000,44.0900,43.960313,0.00,-0.02,...,22.514068,19.172360,0.280,0.2000,0.3700,0.285075,1.0,1.0,1.0,1.0
9,5,0,2398,0.022815,43.99,43.8000,44.0900,43.960313,0.00,-0.02,...,22.514068,19.172360,0.280,0.2000,0.3700,0.285075,1.0,1.0,1.0,1.0


## Short-segment diagnostic

Check the temporal continuity of the retained 2019 pump-state sequence and identify possible blink-like inserted segments.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Diagnostic check for short inserted / blink-like episodes
# for the retained pump-state solution k=4

INPUT_FILE = Path("pump_kmeans_screening_2019/pump_kmeans_labels_k2_12.csv")
OUTPUT_DIR = Path("pump_final_states_2019")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATE_COLUMNS = {
    "k4_main": "pump_state_k4",
}

# Diagnostic thresholds.
# The code reports all of them so we can see whether blink handling is even relevant.
BLINK_THRESHOLDS_MIN = [5, 10, 15, 20, 25, 30]

# If True: duration <= threshold.
# If False: duration < threshold.
USE_LEQ_THRESHOLD = True

EXPECTED_STEP_MIN = None
# If None, the code estimates the dominant SCADA step from Timestamp.
# The expected SCADA step is 5 minutes.


def read_semicolon_csv_robust(path: Path) -> pd.DataFrame:
    """Read a semicolon-separated CSV and parse Timestamp."""
    df = pd.read_csv(path, sep=";", low_memory=False)
    if "Timestamp" not in df.columns:
        raise ValueError("Input file must contain a 'Timestamp' column.")
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    if df["Timestamp"].isna().any():
        bad = df["Timestamp"].isna().sum()
        raise ValueError(f"Timestamp parsing failed for {bad} rows.")
    df = df.sort_values("Timestamp").reset_index(drop=True)
    return df


def infer_sampling_step_minutes(timestamps: pd.Series) -> int:
    """Infer the dominant sampling interval in minutes."""
    diffs = timestamps.diff().dropna()
    if diffs.empty:
        raise ValueError("Cannot infer sampling step from fewer than two timestamps.")
    step = diffs.mode().iloc[0]
    return int(round(step.total_seconds() / 60))


def construct_episodes(df: pd.DataFrame, state_col: str, solution_name: str, step_min: int) -> pd.DataFrame:
    """
    Construct episodes as contiguous runs of identical state labels.
    A new episode starts when:
      1) the state label changes, or
      2) timestamp continuity is interrupted.
    """
    x = df[["Timestamp", state_col]].copy()
    x = x.rename(columns={state_col: "state"})
    x["state"] = x["state"].astype("Int64")

    expected_step = pd.Timedelta(minutes=step_min)

    x["prev_state"] = x["state"].shift(1)
    x["prev_timestamp"] = x["Timestamp"].shift(1)
    x["time_gap"] = x["Timestamp"] - x["prev_timestamp"]

    x["new_episode"] = (
        x["prev_state"].isna()
        | (x["state"] != x["prev_state"])
        | (x["time_gap"] != expected_step)
    )

    x["episode_number"] = x["new_episode"].cumsum().astype(int)

    episodes = (
        x.groupby("episode_number", as_index=False)
        .agg(
            state=("state", "first"),
            start_time=("Timestamp", "first"),
            end_time=("Timestamp", "last"),
            n_timestamps=("Timestamp", "size"),
        )
    )

    episodes.insert(0, "solution", solution_name)
    episodes["episode_id"] = (
        episodes["solution"]
        + "_E"
        + episodes["episode_number"].astype(str).str.zfill(6)
    )

    # For regular 5-min timestamped SCADA data, duration as coverage is n_timestamps * step.
    # This matches the interpretation that 6 observations at 5-min resolution cover 30 min.
    episodes["duration_min"] = episodes["n_timestamps"] * step_min
    episodes["duration_h"] = episodes["duration_min"] / 60.0

    # Neighbouring episode information
    episodes["prev_state"] = episodes["state"].shift(1)
    episodes["next_state"] = episodes["state"].shift(-1)
    episodes["prev_episode_id"] = episodes["episode_id"].shift(1)
    episodes["next_episode_id"] = episodes["episode_id"].shift(-1)
    episodes["prev_end_time"] = episodes["end_time"].shift(1)
    episodes["next_start_time"] = episodes["start_time"].shift(-1)

    episodes["gap_from_prev_min"] = (
        (episodes["start_time"] - episodes["prev_end_time"])
        .dt.total_seconds()
        .div(60)
    )
    episodes["gap_to_next_min"] = (
        (episodes["next_start_time"] - episodes["end_time"])
        .dt.total_seconds()
        .div(60)
    )

    # Structural inserted episode:
    # previous and next episodes exist, have the same state, and the current state differs.
    episodes["is_inserted_between_same_state"] = (
        episodes["prev_state"].notna()
        & episodes["next_state"].notna()
        & (episodes["prev_state"] == episodes["next_state"])
        & (episodes["state"] != episodes["prev_state"])
    )

    # Timestamp continuity around the inserted episode.
    episodes["has_regular_neighbour_continuity"] = (
        (episodes["gap_from_prev_min"] == step_min)
        & (episodes["gap_to_next_min"] == step_min)
    )

    episodes["is_structural_blink_candidate"] = (
        episodes["is_inserted_between_same_state"]
        & episodes["has_regular_neighbour_continuity"]
    )

    return episodes


def summarize_blinks_by_threshold(episodes: pd.DataFrame, thresholds_min: list[int], use_leq: bool) -> pd.DataFrame:
    """Count blink candidates under multiple duration thresholds."""
    rows = []

    total_episodes = len(episodes)
    total_timestamps = episodes["n_timestamps"].sum()

    structural = episodes["is_structural_blink_candidate"]

    for thr in thresholds_min:
        if use_leq:
            duration_condition = episodes["duration_min"] <= thr
            rule = f"duration <= {thr} min"
        else:
            duration_condition = episodes["duration_min"] < thr
            rule = f"duration < {thr} min"

        blink = structural & duration_condition

        rows.append(
            {
                "solution": episodes["solution"].iloc[0],
                "threshold_min": thr,
                "threshold_rule": rule,
                "total_episodes": total_episodes,
                "blink_candidate_episodes": int(blink.sum()),
                "blink_candidate_episode_share_pct": 100 * blink.sum() / total_episodes if total_episodes else np.nan,
                "blink_candidate_timestamps": int(episodes.loc[blink, "n_timestamps"].sum()),
                "blink_candidate_timestamp_share_pct": (
                    100 * episodes.loc[blink, "n_timestamps"].sum() / total_timestamps
                    if total_timestamps else np.nan
                ),
                "median_blink_duration_min": (
                    episodes.loc[blink, "duration_min"].median()
                    if blink.any() else np.nan
                ),
                "max_blink_duration_min": (
                    episodes.loc[blink, "duration_min"].max()
                    if blink.any() else np.nan
                ),
            }
        )

    return pd.DataFrame(rows)


labels = read_semicolon_csv_robust(INPUT_FILE)

if EXPECTED_STEP_MIN is None:
    step_min = infer_sampling_step_minutes(labels["Timestamp"])
else:
    step_min = EXPECTED_STEP_MIN

print(f"Detected sampling step: {step_min} min")
print(f"Rows in label file: {len(labels):,}")
print(f"Time span: {labels['Timestamp'].min()} to {labels['Timestamp'].max()}")

all_episodes = []
all_blink_summaries = []

for solution_name, state_col in STATE_COLUMNS.items():
    if state_col not in labels.columns:
        raise ValueError(f"Required column '{state_col}' not found in {INPUT_FILE}")

    episodes = construct_episodes(
        df=labels,
        state_col=state_col,
        solution_name=solution_name,
        step_min=step_min,
    )

    blink_summary = summarize_blinks_by_threshold(
        episodes=episodes,
        thresholds_min=BLINK_THRESHOLDS_MIN,
        use_leq=USE_LEQ_THRESHOLD,
    )

    all_episodes.append(episodes)
    all_blink_summaries.append(blink_summary)

episodes_all = pd.concat(all_episodes, ignore_index=True)
blink_summary_all = pd.concat(all_blink_summaries, ignore_index=True)

# Add threshold-specific flags to the episode table for convenience
for thr in BLINK_THRESHOLDS_MIN:
    if USE_LEQ_THRESHOLD:
        episodes_all[f"is_blink_candidate_le_{thr}min"] = (
            episodes_all["is_structural_blink_candidate"]
            & (episodes_all["duration_min"] <= thr)
        )
    else:
        episodes_all[f"is_blink_candidate_lt_{thr}min"] = (
            episodes_all["is_structural_blink_candidate"]
            & (episodes_all["duration_min"] < thr)
        )

episodes_out = OUTPUT_DIR / "PUMP_1_blink_diagnostic_episodes_k4_main_2019.csv"
summary_out = OUTPUT_DIR / "PUMP_1_blink_diagnostic_summary_k4_main_2019.csv"
candidates_out = OUTPUT_DIR / "PUMP_1_blink_diagnostic_candidates_k4_main_2019.csv"

episodes_all.to_csv(episodes_out, sep=";", index=False)
blink_summary_all.to_csv(summary_out, sep=";", index=False)

# Save detailed candidates at the most relevant diagnostic threshold, usually 15 min
main_threshold = 15
candidate_col = (
    f"is_blink_candidate_le_{main_threshold}min"
    if USE_LEQ_THRESHOLD
    else f"is_blink_candidate_lt_{main_threshold}min"
)
blink_candidates_15 = episodes_all.loc[episodes_all[candidate_col]].copy()
blink_candidates_15.to_csv(candidates_out, sep=";", index=False)

# Display compact results
print("\nBlink-candidate summary by threshold:")
display_cols = [
    "solution",
    "threshold_rule",
    "total_episodes",
    "blink_candidate_episodes",
    "blink_candidate_episode_share_pct",
    "blink_candidate_timestamps",
    "blink_candidate_timestamp_share_pct",
    "median_blink_duration_min",
    "max_blink_duration_min",
]
display(
    blink_summary_all[display_cols].style.format(
        {
            "blink_candidate_episode_share_pct": "{:.3f}",
            "blink_candidate_timestamp_share_pct": "{:.5f}",
            "median_blink_duration_min": "{:.1f}",
            "max_blink_duration_min": "{:.1f}",
        }
    )
)


Detected sampling step: 5 min
Rows in label file: 105,108
Time span: 2019-01-01 01:00:00 to 2019-12-31 23:55:00

Blink-candidate summary by threshold:


,solution,threshold_rule,total_episodes,blink_candidate_episodes,blink_candidate_episode_share_pct,blink_candidate_timestamps,blink_candidate_timestamp_share_pct,median_blink_duration_min,max_blink_duration_min
0,k4_main,duration <= 5 min,873,0,0.000,0,0.00000,nan,nan
1,k4_main,duration <= 10 min,873,0,0.000,0,0.00000,nan,nan
2,k4_main,duration <= 15 min,873,0,0.000,0,0.00000,nan,nan
3,k4_main,duration <= 20 min,873,0,0.000,0,0.00000,nan,nan
4,k4_main,duration <= 25 min,873,0,0.000,0,0.00000,nan,nan
5,k4_main,duration <= 30 min,873,0,0.000,0,0.00000,nan,nan


## Pump-state episode profiling

Summarize the retained 2019 pump-state profiles and contiguous state-segment statistics.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

LABEL_FILE = Path("pump_kmeans_screening_2019/pump_kmeans_labels_k2_12.csv")
PUMP_FEATURE_FILE = Path("pump_features_2019/PUMP_1_pump_features_2019.csv")

OUTPUT_DIR = Path("pump_final_states_2019")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATE_COLUMNS = {
    "k4_main": "pump_state_k4",
}

# Blink diagnostic thresholds. These are only flags, not corrections.
BLINK_THRESHOLDS_MIN = [15, 30]

# Interpretation labels for the retained k=4 solution.
STATE_INTERPRETATION = {
    ("k4_main", 0): "Dominant stable pump-flow context",
    ("k4_main", 1): "Post-increase / recent high-flow context",
    ("k4_main", 2): "Low-flow / decreasing-flow transition context",
    ("k4_main", 3): "Increasing-flow transition context",
}

# Robust reading helpers
def read_semicolon_csv_robust(path: Path) -> pd.DataFrame:
    """
    Reads a semicolon-separated CSV and robustly converts decimal commas to decimal points.
    Timestamp is parsed if present.
    """

    df = pd.read_csv(path, sep=";", dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
        if df["Timestamp"].isna().any():
            n_bad = int(df["Timestamp"].isna().sum())
            raise ValueError(f"Timestamp parsing failed for {n_bad} rows in {path}")

    for col in df.columns:
        if col == "Timestamp":
            continue

        s = df[col].astype(str).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
        s = s.str.replace(",", ".", regex=False)

        converted = pd.to_numeric(s, errors="coerce")

        # Convert column if at least some values are numeric.
        if converted.notna().sum() > 0:
            df[col] = converted
        else:
            df[col] = df[col]

    return df


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Finds the first available column from a list of possible names.
    Matching is case-insensitive.
    """
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def infer_step_min(timestamps: pd.Series) -> int:
    diffs = timestamps.sort_values().diff().dropna()
    if diffs.empty:
        raise ValueError("Cannot infer sampling step from fewer than two timestamps.")
    step = diffs.mode().iloc[0]
    return int(round(step.total_seconds() / 60))


# Read inputs
labels = read_semicolon_csv_robust(LABEL_FILE)
features = read_semicolon_csv_robust(PUMP_FEATURE_FILE)

if "Timestamp" not in labels.columns:
    raise ValueError("The label file must contain a Timestamp column.")
if "Timestamp" not in features.columns:
    raise ValueError("The pump feature file must contain a Timestamp column.")

labels = labels.sort_values("Timestamp").reset_index(drop=True)
features = features.sort_values("Timestamp").reset_index(drop=True)

for solution, state_col in STATE_COLUMNS.items():
    if state_col not in labels.columns:
        raise ValueError(f"Required state column not found in label file: {state_col}")

step_min = infer_step_min(labels["Timestamp"])

print(f"Detected sampling step: {step_min} min")
print(f"Labels:   {len(labels):,} rows, {labels['Timestamp'].min()} to {labels['Timestamp'].max()}")
print(f"Features: {len(features):,} rows, {features['Timestamp'].min()} to {features['Timestamp'].max()}")

# Harmonize feature column names
feature_aliases = {
    "q_p": ["q_p", "PUMP_1", "pump_flow", "pump_link_flow", "q"],
    "delta_q_p_5min": ["delta_q_p_5min", "delta_q_5min", "dq_5min"],
    "delta_q_p_30min": ["delta_q_p_30min", "delta_q_30min", "dq_30min"],
    "delta_q_p_60min": ["delta_q_p_60min", "delta_q_60min", "dq_60min"],
    "std_q_p_30min": ["std_q_p_30min", "std_q_30min"],
    "std_q_p_60min": ["std_q_p_60min", "std_q_60min"],
    "d_q_p_tod_pm30min": [
        "d_q_p_tod_pm30min",
        "q_p_tod_deviation_pm30min",
        "q_tod_deviation_pm30min",
        "d_q_tod_pm30min",
    ],
    "pump_activity": ["pump_activity", "q_regime_active", "pump_regime_active"],
}

rename_map = {}
for canonical, candidates in feature_aliases.items():
    found = find_column(features, candidates)
    if found is not None:
        rename_map[found] = canonical

features = features.rename(columns=rename_map)

profile_feature_cols = [
    c for c in [
        "q_p",
        "delta_q_p_5min",
        "delta_q_p_30min",
        "delta_q_p_60min",
        "std_q_p_30min",
        "std_q_p_60min",
        "d_q_p_tod_pm30min",
        "pump_activity",
    ]
    if c in features.columns
]

if not profile_feature_cols:
    raise ValueError(
        "No recognizable pump feature columns were found. "
        "Check column names in PUMP_1_pump_features_2019.csv."
    )

print("\nPump features used for episode profiling:")
print(profile_feature_cols)

# Merge labels with pump features
base = labels[["Timestamp"] + list(STATE_COLUMNS.values())].merge(
    features[["Timestamp"] + profile_feature_cols],
    on="Timestamp",
    how="left",
    validate="one_to_one",
)

missing_feature_rows = int(base[profile_feature_cols].isna().all(axis=1).sum())
if missing_feature_rows > 0:
    print(f"\nWarning: {missing_feature_rows:,} label rows have no matching pump features.")

# Episode construction
def assign_episode_ids(df: pd.DataFrame, state_col: str, solution: str, step_min: int) -> pd.DataFrame:
    """
    Creates timestamp-level episode IDs for one state solution.
    New episode starts after state change or timestamp continuity interruption.
    """
    x = df[["Timestamp", state_col]].copy()
    x = x.rename(columns={state_col: "state"})

    expected_step = pd.Timedelta(minutes=step_min)

    x["prev_state"] = x["state"].shift(1)
    x["prev_time"] = x["Timestamp"].shift(1)
    x["time_gap"] = x["Timestamp"] - x["prev_time"]

    x["new_episode"] = (
        x["prev_state"].isna()
        | (x["state"] != x["prev_state"])
        | (x["time_gap"] != expected_step)
    )

    x["episode_number"] = x["new_episode"].cumsum().astype(int)
    x["episode_id"] = solution + "_E" + x["episode_number"].astype(str).str.zfill(6)

    return x[["Timestamp", "state", "episode_number", "episode_id"]]


def build_episode_profiles(
    timestamp_data: pd.DataFrame,
    solution: str,
    state_col: str,
    profile_cols: list[str],
    step_min: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    assigned = assign_episode_ids(timestamp_data, state_col, solution, step_min)

    tmp = timestamp_data.merge(
        assigned,
        on="Timestamp",
        how="left",
        validate="one_to_one",
    )

    # Timestamp-level state/episode sequence for later baseline and residual work
    sequence = tmp[["Timestamp", "state", "episode_id"]].copy()
    sequence = sequence.rename(
        columns={
            "state": f"pump_state_{solution}",
            "episode_id": f"pump_episode_id_{solution}",
        }
    )

    episode_rows = []

    for episode_id, g in tmp.groupby("episode_id", sort=False):
        row = {
            "solution": solution,
            "episode_id": episode_id,
            "episode_number": int(g["episode_number"].iloc[0]),
            "state": int(g["state"].iloc[0]),
            "state_label": f"S{int(g['state'].iloc[0])}",
            "start_time": g["Timestamp"].min(),
            "end_time": g["Timestamp"].max(),
            "n_timestamps": int(len(g)),
            "duration_min": int(len(g) * step_min),
            "duration_h": len(g) * step_min / 60.0,
        }

        for col in profile_cols:
            values = pd.to_numeric(g[col], errors="coerce")
            row[f"{col}_missing_share"] = values.isna().mean()

            if values.notna().sum() > 0:
                row[f"{col}_mean"] = values.mean()
                row[f"{col}_median"] = values.median()
                row[f"{col}_q10"] = values.quantile(0.10)
                row[f"{col}_q90"] = values.quantile(0.90)
                row[f"{col}_min"] = values.min()
                row[f"{col}_max"] = values.max()
                row[f"{col}_std"] = values.std(ddof=0)
            else:
                row[f"{col}_mean"] = np.nan
                row[f"{col}_median"] = np.nan
                row[f"{col}_q10"] = np.nan
                row[f"{col}_q90"] = np.nan
                row[f"{col}_min"] = np.nan
                row[f"{col}_max"] = np.nan
                row[f"{col}_std"] = np.nan

        # More readable activity share if pump_activity is available
        if "pump_activity" in profile_cols:
            activity = pd.to_numeric(g["pump_activity"], errors="coerce")
            row["pump_activity_share"] = activity.mean()

        row["interpretation"] = STATE_INTERPRETATION.get(
            (solution, int(g["state"].iloc[0])),
            "",
        )

        episode_rows.append(row)

    episodes = pd.DataFrame(episode_rows)

    # Neighbouring episode structure for blink diagnostics
    episodes["prev_state"] = episodes["state"].shift(1)
    episodes["next_state"] = episodes["state"].shift(-1)
    episodes["prev_episode_id"] = episodes["episode_id"].shift(1)
    episodes["next_episode_id"] = episodes["episode_id"].shift(-1)
    episodes["prev_end_time"] = episodes["end_time"].shift(1)
    episodes["next_start_time"] = episodes["start_time"].shift(-1)

    episodes["gap_from_prev_min"] = (
        (episodes["start_time"] - episodes["prev_end_time"])
        .dt.total_seconds()
        .div(60)
    )
    episodes["gap_to_next_min"] = (
        (episodes["next_start_time"] - episodes["end_time"])
        .dt.total_seconds()
        .div(60)
    )

    episodes["is_inserted_between_same_state"] = (
        episodes["prev_state"].notna()
        & episodes["next_state"].notna()
        & (episodes["prev_state"] == episodes["next_state"])
        & (episodes["state"] != episodes["prev_state"])
    )

    episodes["has_regular_neighbour_continuity"] = (
        (episodes["gap_from_prev_min"] == step_min)
        & (episodes["gap_to_next_min"] == step_min)
    )

    episodes["is_structural_blink_candidate"] = (
        episodes["is_inserted_between_same_state"]
        & episodes["has_regular_neighbour_continuity"]
    )

    for thr in BLINK_THRESHOLDS_MIN:
        episodes[f"is_blink_candidate_le_{thr}min"] = (
            episodes["is_structural_blink_candidate"]
            & (episodes["duration_min"] <= thr)
        )

    return sequence, episodes


# Build profiles for retained k=4
all_sequences = []
all_episode_profiles = []

for solution, state_col in STATE_COLUMNS.items():
    seq, ep = build_episode_profiles(
        timestamp_data=base,
        solution=solution,
        state_col=state_col,
        profile_cols=profile_feature_cols,
        step_min=step_min,
    )
    all_sequences.append(seq)
    all_episode_profiles.append(ep)

episode_profiles = pd.concat(all_episode_profiles, ignore_index=True)

# Merge timestamp-level sequences into one wide table
final_sequence = base[["Timestamp"]].copy()
for seq in all_sequences:
    final_sequence = final_sequence.merge(seq, on="Timestamp", how="left", validate="one_to_one")

# Add basic pump features to timestamp sequence for convenience
final_sequence = final_sequence.merge(
    features[["Timestamp"] + profile_feature_cols],
    on="Timestamp",
    how="left",
    validate="one_to_one",
)

# State-level profiles derived from episode profiles
state_rows = []

for (solution, state), eps in episode_profiles.groupby(["solution", "state"], sort=True):
    total_timestamps_solution = episode_profiles.loc[
        episode_profiles["solution"] == solution,
        "n_timestamps"
    ].sum()

    row = {
        "solution": solution,
        "state": int(state),
        "state_label": f"S{int(state)}",
        "interpretation": STATE_INTERPRETATION.get((solution, int(state)), ""),
        "episodes": int(len(eps)),
        "timestamps": int(eps["n_timestamps"].sum()),
        "share_timestamps_pct": 100 * eps["n_timestamps"].sum() / total_timestamps_solution,
        "median_episode_duration_min": eps["duration_min"].median(),
        "mean_episode_duration_min": eps["duration_min"].mean(),
        "min_episode_duration_min": eps["duration_min"].min(),
        "max_episode_duration_min": eps["duration_min"].max(),
        "episodes_lt_30min": int((eps["duration_min"] < 30).sum()),
        "episodes_le_15min": int((eps["duration_min"] <= 15).sum()),
        "blink_candidates_le_15min": int(eps.get("is_blink_candidate_le_15min", pd.Series(False, index=eps.index)).sum()),
        "blink_candidates_le_30min": int(eps.get("is_blink_candidate_le_30min", pd.Series(False, index=eps.index)).sum()),
    }

    # State-level medians of episode-level medians
    for col in profile_feature_cols:
        med_col = f"{col}_median"
        if med_col in eps.columns:
            row[f"median_{col}"] = eps[med_col].median()

    if "pump_activity_share" in eps.columns:
        row["median_pump_activity_share"] = eps["pump_activity_share"].median()

    state_rows.append(row)

state_profiles = pd.DataFrame(state_rows)

# Solution-level summary
solution_rows = []
for solution, eps in episode_profiles.groupby("solution", sort=True):
    row = {
        "solution": solution,
        "episodes": int(len(eps)),
        "timestamps": int(eps["n_timestamps"].sum()),
        "median_episode_duration_min": eps["duration_min"].median(),
        "mean_episode_duration_min": eps["duration_min"].mean(),
        "min_episode_duration_min": eps["duration_min"].min(),
        "max_episode_duration_min": eps["duration_min"].max(),
        "episodes_lt_30min": int((eps["duration_min"] < 30).sum()),
        "episodes_lt_30min_pct": 100 * (eps["duration_min"] < 30).mean(),
        "blink_candidates_le_15min": int(eps["is_blink_candidate_le_15min"].sum()),
        "blink_candidates_le_15min_pct": 100 * eps["is_blink_candidate_le_15min"].mean(),
        "blink_candidates_le_30min": int(eps["is_blink_candidate_le_30min"].sum()),
        "blink_candidates_le_30min_pct": 100 * eps["is_blink_candidate_le_30min"].mean(),
    }
    solution_rows.append(row)

solution_summary = pd.DataFrame(solution_rows)

sequence_out = OUTPUT_DIR / "PUMP_1_final_state_sequence_k4_main_2019.csv"
episode_profiles_out = OUTPUT_DIR / "PUMP_1_final_episode_profiles_k4_main_2019.csv"
state_profiles_out = OUTPUT_DIR / "PUMP_1_final_state_profiles_k4_main_2019.csv"
solution_summary_out = OUTPUT_DIR / "PUMP_1_final_episode_summary_by_solution_k4_main_2019.csv"

final_sequence.to_csv(sequence_out, sep=";", index=False)
episode_profiles.to_csv(episode_profiles_out, sep=";", index=False)
state_profiles.to_csv(state_profiles_out, sep=";", index=False)
solution_summary.to_csv(solution_summary_out, sep=";", index=False)

# Display compact outputs
print(f"- Timestamp-level state sequence: {sequence_out}")
print(f"- Episode profiles:              {episode_profiles_out}")
print(f"- State profiles:                {state_profiles_out}")
print(f"- Solution summary:              {solution_summary_out}")

print("\nSolution-level episode summary:")
display(
    solution_summary.style.format({
        "median_episode_duration_min": "{:.1f}",
        "mean_episode_duration_min": "{:.1f}",
        "episodes_lt_30min_pct": "{:.2f}",
        "blink_candidates_le_15min_pct": "{:.2f}",
        "blink_candidates_le_30min_pct": "{:.2f}",
    })
)

print("\nState-level profiles:")
display_cols = [
    "solution",
    "state_label",
    "interpretation",
    "share_timestamps_pct",
    "episodes",
    "median_episode_duration_min",
]

for col in [
    "median_q_p",
    "median_delta_q_p_30min",
    "median_delta_q_p_60min",
    "median_std_q_p_30min",
    "median_std_q_p_60min",
    "median_d_q_p_tod_pm30min",
    "median_pump_activity_share",
]:
    if col in state_profiles.columns:
        display_cols.append(col)

display_cols += [
    "blink_candidates_le_15min",
    "blink_candidates_le_30min",
]

display(
    state_profiles[display_cols].style.format({
        "share_timestamps_pct": "{:.2f}",
        "median_episode_duration_min": "{:.1f}",
        "median_q_p": "{:.2f}",
        "median_delta_q_p_30min": "{:.2f}",
        "median_delta_q_p_60min": "{:.2f}",
        "median_std_q_p_30min": "{:.2f}",
        "median_std_q_p_60min": "{:.2f}",
        "median_d_q_p_tod_pm30min": "{:.2f}",
        "median_pump_activity_share": "{:.2f}",
    })
)

print("\nFirst rows of final timestamp-level sequence:")
display(final_sequence.head())

print("\nFirst rows of episode profiles:")
display(episode_profiles.head())


Detected sampling step: 5 min
Labels:   105,108 rows, 2019-01-01 01:00:00 to 2019-12-31 23:55:00
Features: 105,120 rows, 2019-01-01 00:00:00 to 2019-12-31 23:55:00

Pump features used for episode profiling:
['q_p', 'delta_q_p_5min', 'delta_q_p_30min', 'delta_q_p_60min', 'std_q_p_30min', 'std_q_p_60min', 'd_q_p_tod_pm30min', 'pump_activity']

Saved outputs:
- Timestamp-level state sequence: pump_final_states_2019\PUMP_1_final_state_sequence_k4_main_2019.csv
- Episode profiles:              pump_final_states_2019\PUMP_1_final_episode_profiles_k4_main_2019.csv
- State profiles:                pump_final_states_2019\PUMP_1_final_state_profiles_k4_main_2019.csv
- Solution summary:              pump_final_states_2019\PUMP_1_final_episode_summary_by_solution_k4_main_2019.csv

Solution-level episode summary:


,solution,episodes,timestamps,median_episode_duration_min,mean_episode_duration_min,min_episode_duration_min,max_episode_duration_min,episodes_lt_30min,episodes_lt_30min_pct,blink_candidates_le_15min,blink_candidates_le_15min_pct,blink_candidates_le_30min,blink_candidates_le_30min_pct
0,k4_main,873,105108,160.0,602.0,55,7015,0,0.00,0,0.00,0,0.00



State-level profiles:


,solution,state_label,interpretation,share_timestamps_pct,episodes,median_episode_duration_min,median_q_p,median_delta_q_p_30min,median_delta_q_p_60min,median_std_q_p_30min,median_std_q_p_60min,median_d_q_p_tod_pm30min,median_pump_activity_share,blink_candidates_le_15min,blink_candidates_le_30min
0,k4_main,S0,Dominant stable pump-flow context,75.62,219,1415.0,43.84,0.00,0.00,0.03,0.03,0.10,1.00,0,0
1,k4_main,S1,Post-increase / recent high-flow context,2.28,218,55.0,0.00,-43.81,-43.84,0.00,19.83,-43.89,0.00,0,0
2,k4_main,S2,Low-flow / decreasing-flow transition context,19.82,218,465.0,0.00,0.00,0.00,0.00,0.00,-43.60,0.00,0,0
3,k4_main,S3,Increasing-flow transition context,2.28,218,55.0,43.99,43.95,43.99,0.04,19.90,0.28,1.00,0,0



First rows of final timestamp-level sequence:


,Timestamp,pump_state_k4_main,pump_episode_id_k4_main,q_p,delta_q_p_5min,delta_q_p_30min,delta_q_p_60min,std_q_p_30min,std_q_p_60min,d_q_p_tod_pm30min,pump_activity
0,2019-01-01 01:00:00,0,k4_main_E000001,44.05,-0.01,-0.01,0.03,0.008944,0.013817,0.10,1
1,2019-01-01 01:05:00,0,k4_main_E000001,44.06,0.01,0.02,0.02,0.008165,0.014355,0.11,1
2,2019-01-01 01:10:00,0,k4_main_E000001,44.05,-0.01,0.00,0.01,0.008165,0.014222,0.09,1
3,2019-01-01 01:15:00,0,k4_main_E000001,44.06,0.01,0.02,0.01,0.005164,0.014668,0.10,1
4,2019-01-01 01:20:00,0,k4_main_E000001,44.06,0.00,0.00,0.05,0.005164,0.008660,0.09,1



First rows of episode profiles:


,solution,episode_id,episode_number,state,state_label,start_time,end_time,n_timestamps,duration_min,duration_h,...,next_episode_id,prev_end_time,next_start_time,gap_from_prev_min,gap_to_next_min,is_inserted_between_same_state,has_regular_neighbour_continuity,is_structural_blink_candidate,is_blink_candidate_le_15min,is_blink_candidate_le_30min
0,k4_main,k4_main_E000001,1,0,S0,2019-01-01 01:00:00,2019-01-01 03:35:00,32,160,2.666667,...,k4_main_E000002,NaT,2019-01-01 03:40:00,NaN,5.0,False,False,False,False,False
1,k4_main,k4_main_E000002,2,1,S1,2019-01-01 03:40:00,2019-01-01 04:30:00,11,55,0.916667,...,k4_main_E000003,2019-01-01 03:35:00,2019-01-01 04:35:00,5.0,5.0,False,True,False,False,False
2,k4_main,k4_main_E000003,3,2,S2,2019-01-01 04:35:00,2019-01-01 13:55:00,113,565,9.416667,...,k4_main_E000004,2019-01-01 04:30:00,2019-01-01 14:00:00,5.0,5.0,False,True,False,False,False
3,k4_main,k4_main_E000004,4,3,S3,2019-01-01 14:00:00,2019-01-01 14:50:00,11,55,0.916667,...,k4_main_E000005,2019-01-01 13:55:00,2019-01-01 14:55:00,5.0,5.0,False,True,False,False,False
4,k4_main,k4_main_E000005,5,0,S0,2019-01-01 14:55:00,2019-01-02 03:25:00,151,755,12.583333,...,k4_main_E000006,2019-01-01 14:50:00,2019-01-02 03:30:00,5.0,5.0,False,True,False,False,False


## Flow-deviation assessment

Compute 2019 timestamp-level flow deviations for p235 and p227 under the four baseline variants.

In [8]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

FLOW_FILE = Path("2019_SCADA_Flows.csv")
LABEL_FILE = Path("pump_kmeans_screening_2019/pump_kmeans_labels_k2_12.csv")

OUTPUT_DIR = Path("flow_baselines_2019")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATE_COL = "pump_state_k4"          # main retained solution
TIMESTAMP_COL = "Timestamp"
FLOW_SIGNALS = ["p235", "p227"]      # non-pump pipe-flow signals only

SEP = ";"
TOD_NEIGHBOUR_SLOTS = 6              # +/- 30 min on 5-min grid
SLOTS_PER_DAY = 288                  # 24 * 60 / 5


# Robust semicolon CSV reader
def read_semicolon_csv_robust(path: Path, timestamp_col: str = "Timestamp") -> pd.DataFrame:

    df = pd.read_csv(path, sep=SEP, dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found in {path}. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    if df[timestamp_col].isna().any():
        n_bad = int(df[timestamp_col].isna().sum())
        raise ValueError(f"Timestamp parsing failed for {n_bad} rows in {path}")

    for col in df.columns:
        if col == timestamp_col:
            continue

        s = df[col].astype(str).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
        s = s.str.replace(",", ".", regex=False)

        converted = pd.to_numeric(s, errors="coerce")

        # keep numeric conversion only if it produced some non-NaN values
        if converted.notna().any():
            df[col] = converted
        else:
            df[col] = s

    df = df.sort_values(timestamp_col).reset_index(drop=True)
    return df


# Cyclic time-of-day baseline
# For each 5-min slot s, compute the median over slots in
# [s-k, ..., s, ..., s+k] modulo 288.
def compute_tod_median_baseline(values: pd.Series, slot_index: pd.Series, neighbour_slots: int = 6) -> pd.Series:
    out = pd.Series(index=values.index, dtype=float)

    # Pre-split values by slot
    by_slot = {s: values[slot_index == s].dropna().to_numpy() for s in range(SLOTS_PER_DAY)}

    slot_medians = {}
    for s in range(SLOTS_PER_DAY):
        neighbour_values = []
        for off in range(-neighbour_slots, neighbour_slots + 1):
            s2 = (s + off) % SLOTS_PER_DAY
            arr = by_slot[s2]
            if arr.size:
                neighbour_values.append(arr)

        if neighbour_values:
            pooled = np.concatenate(neighbour_values)
            slot_medians[s] = float(np.median(pooled))
        else:
            slot_medians[s] = np.nan

    out = slot_index.map(slot_medians).astype(float)
    return out


# Pump-state + time-of-day baseline
# Same cyclic TOD logic, but only within the current pump state.
def compute_state_tod_median_baseline(
    values: pd.Series,
    state_labels: pd.Series,
    slot_index: pd.Series,
    neighbour_slots: int = 6
) -> pd.Series:
    out = pd.Series(index=values.index, dtype=float)

    unique_states = sorted([x for x in state_labels.dropna().unique()])
    slot_state_medians = {}

    for st in unique_states:
        mask_st = (state_labels == st)
        values_st = values[mask_st]
        slots_st = slot_index[mask_st]

        by_slot = {s: values_st[slots_st == s].dropna().to_numpy() for s in range(SLOTS_PER_DAY)}

        for s in range(SLOTS_PER_DAY):
            neighbour_values = []
            for off in range(-neighbour_slots, neighbour_slots + 1):
                s2 = (s + off) % SLOTS_PER_DAY
                arr = by_slot[s2]
                if arr.size:
                    neighbour_values.append(arr)

            if neighbour_values:
                pooled = np.concatenate(neighbour_values)
                slot_state_medians[(st, s)] = float(np.median(pooled))
            else:
                slot_state_medians[(st, s)] = np.nan

    keys = list(zip(state_labels, slot_index))
    out = pd.Series([slot_state_medians.get(k, np.nan) for k in keys], index=values.index, dtype=float)
    return out


# Load and prepare data
flows = read_semicolon_csv_robust(FLOW_FILE, timestamp_col=TIMESTAMP_COL)
labels = read_semicolon_csv_robust(LABEL_FILE, timestamp_col=TIMESTAMP_COL)

required_cols = [TIMESTAMP_COL, STATE_COL]
for c in required_cols:
    if c not in labels.columns:
        raise ValueError(f"Required column '{c}' not found in {LABEL_FILE}")

missing_flow_cols = [c for c in FLOW_SIGNALS if c not in flows.columns]
if missing_flow_cols:
    raise ValueError(
        f"Missing non-pump pipe-flow columns in {FLOW_FILE}: {missing_flow_cols}. "
        f"Available columns: {list(flows.columns)}"
    )

# Keep only what we need
flows_use = flows[[TIMESTAMP_COL] + FLOW_SIGNALS].copy()
labels_use = labels[[TIMESTAMP_COL, STATE_COL]].copy()

# Merge on common timestamps
df = flows_use.merge(labels_use, on=TIMESTAMP_COL, how="inner")
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

# 5-min slot index (0..287)
df["tod_slot_5min"] = ((df[TIMESTAMP_COL].dt.hour * 60 + df[TIMESTAMP_COL].dt.minute) // 5).astype(int)

# Basic checks
if df[STATE_COL].isna().any():
    raise ValueError(f"State column '{STATE_COL}' contains missing values after merge.")

display(df.head())
print(f"Merged rows: {len(df):,}")


# Compute baselines and deviations for each non-pump pipe-flow signal
summary_rows = []

for flow_col in FLOW_SIGNALS:
    q = df[flow_col].astype(float)

    # --- Global baseline
    m_G = float(q.median())
    df[f"{flow_col}_baseline_G"] = m_G

    # --- Time-of-day baseline
    df[f"{flow_col}_baseline_T"] = compute_tod_median_baseline(
        values=q,
        slot_index=df["tod_slot_5min"],
        neighbour_slots=TOD_NEIGHBOUR_SLOTS,
    )

    # --- Pump-state baseline
    state_medians = q.groupby(df[STATE_COL]).median().to_dict()
    df[f"{flow_col}_baseline_S"] = df[STATE_COL].map(state_medians).astype(float)

    # --- Pump-state + time-of-day baseline
    df[f"{flow_col}_baseline_ST"] = compute_state_tod_median_baseline(
        values=q,
        state_labels=df[STATE_COL],
        slot_index=df["tod_slot_5min"],
        neighbour_slots=TOD_NEIGHBOUR_SLOTS,
    )

    # --- Signed deviations
    for b in ["G", "T", "S", "ST"]:
        base_col = f"{flow_col}_baseline_{b}"
        dev_col = f"{flow_col}_dev_{b}"
        abs_col = f"{flow_col}_absdev_{b}"

        df[dev_col] = df[flow_col] - df[base_col]
        df[abs_col] = df[dev_col].abs()

    # --- Simple summary
    for b in ["G", "T", "S", "ST"]:
        summary_rows.append({
            "flow_signal": flow_col,
            "baseline": b,
            "n_rows": int(df[f"{flow_col}_absdev_{b}"].notna().sum()),
            "median_absdev": float(df[f"{flow_col}_absdev_{b}"].median()),
            "p90_absdev": float(df[f"{flow_col}_absdev_{b}"].quantile(0.90)),
            "p95_absdev": float(df[f"{flow_col}_absdev_{b}"].quantile(0.95)),
            "mean_absdev": float(df[f"{flow_col}_absdev_{b}"].mean()),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


timestamp_output = OUTPUT_DIR / "flow_timestamp_baselines_and_deviations_k4_main.csv"
summary_output = OUTPUT_DIR / "flow_baseline_deviation_summary_k4_main.csv"

df.to_csv(timestamp_output, sep=";", index=False)
summary_df.to_csv(summary_output, sep=";", index=False)


,Timestamp,p235,p227,pump_state_k4,tod_slot_5min
0,2019-01-01 01:00:00,76.51,74.14,0,12
1,2019-01-01 01:05:00,71.49,70.75,0,13
2,2019-01-01 01:10:00,73.05,71.05,0,14
3,2019-01-01 01:15:00,68.91,68.03,0,15
4,2019-01-01 01:20:00,70.37,69.19,0,16


Merged rows: 105,108


,flow_signal,baseline,n_rows,median_absdev,p90_absdev,p95_absdev,mean_absdev
0,p235,G,105108,24.280,54.0830,67.46650,27.576991
1,p235,T,105108,12.830,34.0530,42.40000,15.973734
2,p235,S,105108,20.100,53.4215,63.67000,24.554224
3,p235,ST,105108,10.085,27.3715,32.33000,12.554981
4,p227,G,105108,26.590,56.3800,67.64000,29.545047
5,p227,T,105108,18.710,39.0000,45.49325,20.407831
6,p227,S,105108,23.340,53.9200,64.50000,26.427598
7,p227,ST,105108,15.210,33.2400,38.31000,16.694756


Saved timestamp-level output to: flow_baselines_2019\flow_timestamp_baselines_and_deviations_k4_main.csv
Saved summary output to:        flow_baselines_2019\flow_baseline_deviation_summary_k4_main.csv


## Leakage-oriented validation of flow deviations

Validate 2019 flow-deviation scores against timestamp-level total leakage intensity.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

DEV_FILE = Path("flow_baselines_2019/flow_timestamp_baselines_and_deviations_k4_main.csv")
LEAK_FILE = Path("2019_Leakages.csv")

TIMESTAMP_COL = "Timestamp"
SEP = ";"

FLOW_SIGNALS = ["p235", "p227"]
BASELINES = ["G", "T", "S", "ST"]

LOW_Q = 0.20
HIGH_Q = 0.80
TOP_PCTS = [0.01, 0.05]
HIGH_LEAK_QUANTILE = 0.80

OUTPUT_DIR = Path("flow_baseline_validation_2019")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Robust semicolon CSV reader
def read_semicolon_csv_robust(path: Path, timestamp_col: str = "Timestamp") -> pd.DataFrame:

    df = pd.read_csv(path, sep=SEP, dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found in {path}. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    if df[timestamp_col].isna().any():
        n_bad = int(df[timestamp_col].isna().sum())
        raise ValueError(f"Timestamp parsing failed for {n_bad} rows in {path}")

    for col in df.columns:
        if col == timestamp_col:
            continue

        s = df[col].astype(str).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
        s = s.str.replace(",", ".", regex=False)

        converted = pd.to_numeric(s, errors="coerce")

        if converted.notna().any():
            df[col] = converted
        else:
            df[col] = s

    df = df.sort_values(timestamp_col).reset_index(drop=True)
    return df


# Helpers
def safe_mean(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.mean()) if len(x) else np.nan

def safe_median(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.median()) if len(x) else np.nan

def top_tail_stats(df_in: pd.DataFrame, score_col: str, leak_col: str, top_pct: float) -> dict:
    df_work = df_in[[score_col, leak_col]].dropna().copy()
    if df_work.empty:
        return {
            f"top_{int(top_pct*100)}pct_n": 0,
            f"top_{int(top_pct*100)}pct_leak_mean": np.nan,
            f"top_{int(top_pct*100)}pct_leak_median": np.nan,
            f"top_{int(top_pct*100)}pct_threshold": np.nan,
            f"top_{int(top_pct*100)}pct_enrichment_ratio": np.nan,
        }

    thr = df_work[score_col].quantile(1 - top_pct)
    top_df = df_work.loc[df_work[score_col] >= thr]

    global_mean = safe_mean(df_work[leak_col])
    top_mean = safe_mean(top_df[leak_col])

    return {
        f"top_{int(top_pct*100)}pct_n": int(len(top_df)),
        f"top_{int(top_pct*100)}pct_leak_mean": top_mean,
        f"top_{int(top_pct*100)}pct_leak_median": safe_median(top_df[leak_col]),
        f"top_{int(top_pct*100)}pct_threshold": float(thr),
        f"top_{int(top_pct*100)}pct_enrichment_ratio": float(top_mean / global_mean) if pd.notna(global_mean) and global_mean != 0 else np.nan,
    }

def low_high_contrast(df_in: pd.DataFrame, score_col: str, leak_col: str, low_q: float = 0.20, high_q: float = 0.80) -> dict:
    df_work = df_in[[score_col, leak_col]].dropna().copy()
    if df_work.empty:
        return {
            "low_thr": np.nan,
            "high_thr": np.nan,
            "low_n": 0,
            "high_n": 0,
            "low_leak_mean": np.nan,
            "high_leak_mean": np.nan,
            "low_leak_median": np.nan,
            "high_leak_median": np.nan,
            "high_low_mean_ratio": np.nan,
        }

    low_thr = df_work[score_col].quantile(low_q)
    high_thr = df_work[score_col].quantile(high_q)

    low_df = df_work.loc[df_work[score_col] <= low_thr]
    high_df = df_work.loc[df_work[score_col] >= high_thr]

    low_mean = safe_mean(low_df[leak_col])
    high_mean = safe_mean(high_df[leak_col])

    return {
        "low_thr": float(low_thr),
        "high_thr": float(high_thr),
        "low_n": int(len(low_df)),
        "high_n": int(len(high_df)),
        "low_leak_mean": low_mean,
        "high_leak_mean": high_mean,
        "low_leak_median": safe_median(low_df[leak_col]),
        "high_leak_median": safe_median(high_df[leak_col]),
        "high_low_mean_ratio": float(high_mean / low_mean) if pd.notna(low_mean) and low_mean != 0 else np.nan,
    }


# Load data
dev = read_semicolon_csv_robust(DEV_FILE, timestamp_col=TIMESTAMP_COL)
leak = read_semicolon_csv_robust(LEAK_FILE, timestamp_col=TIMESTAMP_COL)

# Derive aggregate leakage variables from 2019_Leakages.csv
leak_cols = [c for c in leak.columns if c != TIMESTAMP_COL]

if not leak_cols:
    raise ValueError("No leakage columns found in leakage file.")

leak["total_leak_flow"] = leak[leak_cols].sum(axis=1)
leak["leak_active"] = (leak["total_leak_flow"] > 0).astype(int)
leak["active_leak_count"] = (leak[leak_cols] > 0).sum(axis=1)
leak["max_leak_flow"] = leak[leak_cols].max(axis=1)

nonzero_total = leak.loc[leak["total_leak_flow"] > 0, "total_leak_flow"]
if len(nonzero_total):
    high_thr_global = nonzero_total.quantile(HIGH_LEAK_QUANTILE)
else:
    high_thr_global = 0.0

leak["high_leak_intensity"] = ((leak["total_leak_flow"] >= high_thr_global) & (leak["total_leak_flow"] > 0)).astype(int)

leak_use = leak[
    [
        TIMESTAMP_COL,
        "total_leak_flow",
        "leak_active",
        "active_leak_count",
        "max_leak_flow",
        "high_leak_intensity",
    ]
].copy()

display(leak_use.head())
print(f"High-leak threshold (nonzero total leak, q={HIGH_LEAK_QUANTILE:.2f}): {high_thr_global:.6f}")


# Merge deviations with leakage context
df = dev.merge(leak_use, on=TIMESTAMP_COL, how="inner")
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

print(f"Merged rows for validation: {len(df):,}")
display(df.head())


# Evaluate each flow signal and baseline variant
rows = []

for flow_col in FLOW_SIGNALS:
    for b in BASELINES:
        score_col = f"{flow_col}_absdev_{b}"

        if score_col not in df.columns:
            raise ValueError(f"Missing score column: {score_col}")

        row = {
            "flow_signal": flow_col,
            "baseline": b,
            "n_rows": int(df[score_col].notna().sum()),
            "score_median": safe_median(df[score_col]),
            "score_p90": float(df[score_col].quantile(0.90)),
            "score_p95": float(df[score_col].quantile(0.95)),
        }

        # Low-vs-high contrast on total leak flow
        row.update(low_high_contrast(df, score_col, "total_leak_flow", low_q=LOW_Q, high_q=HIGH_Q))

        # Top-tail enrichment on total leak flow
        for top_pct in TOP_PCTS:
            row.update(top_tail_stats(df, score_col, "total_leak_flow", top_pct=top_pct))

        rows.append(row)

results = pd.DataFrame(rows)

# Helpful ordering
baseline_order = pd.Categorical(results["baseline"], categories=BASELINES, ordered=True)
results = results.assign(_baseline_order=baseline_order).sort_values(
    ["flow_signal", "_baseline_order"]
).drop(columns="_baseline_order").reset_index(drop=True)

display(results)


out_file = OUTPUT_DIR / "flow_leakage_validation_k4_main.csv"
results.to_csv(out_file, sep=";", index=False)


,Timestamp,total_leak_flow,leak_active,active_leak_count,max_leak_flow,high_leak_intensity
0,2019-01-01 00:00:00,24.16,1,4,6.87,0
1,2019-01-01 00:05:00,24.18,1,4,6.87,0
2,2019-01-01 00:10:00,24.18,1,4,6.87,0
3,2019-01-01 00:15:00,24.18,1,4,6.87,0
4,2019-01-01 00:20:00,24.17,1,4,6.87,0


High-leak threshold (nonzero total leak, q=0.80): 103.670000
Merged rows for validation: 105,108


,Timestamp,p235,p227,pump_state_k4,tod_slot_5min,p235_baseline_G,p235_baseline_T,p235_baseline_S,p235_baseline_ST,p235_dev_G,...,p227_absdev_T,p227_dev_S,p227_absdev_S,p227_dev_ST,p227_absdev_ST,total_leak_flow,leak_active,active_leak_count,max_leak_flow,high_leak_intensity
0,2019-01-01 01:00:00,76.51,74.14,0,12,130.12,103.610,140.01,106.585,-53.61,...,29.62,-65.48,65.48,-33.80,33.80,24.23,1,4,6.88,0
1,2019-01-01 01:05:00,71.49,70.75,0,13,130.12,102.010,140.01,105.160,-58.63,...,31.79,-68.87,68.87,-35.77,35.77,24.23,1,4,6.88,0
2,2019-01-01 01:10:00,73.05,71.05,0,14,130.12,100.500,140.01,103.970,-57.07,...,30.27,-68.57,68.57,-33.96,33.96,24.23,1,4,6.88,0
3,2019-01-01 01:15:00,68.91,68.03,0,15,130.12,99.165,140.01,102.720,-61.21,...,32.07,-71.59,71.59,-35.70,35.70,24.24,1,4,6.88,0
4,2019-01-01 01:20:00,70.37,69.19,0,16,130.12,97.910,140.01,101.265,-59.75,...,29.76,-70.43,70.43,-33.24,33.24,24.24,1,4,6.88,0


,flow_signal,baseline,n_rows,score_median,score_p90,score_p95,low_thr,high_thr,low_n,high_n,...,top_1pct_n,top_1pct_leak_mean,top_1pct_leak_median,top_1pct_threshold,top_1pct_enrichment_ratio,top_5pct_n,top_5pct_leak_mean,top_5pct_leak_median,top_5pct_threshold,top_5pct_enrichment_ratio
0,p227,G,105108,26.590,56.3800,67.64000,11.23,45.510,21023,21024,...,1053,30.769069,29.45,89.80000,0.412141,5258,41.917159,30.39,67.64000,0.561465
1,p227,T,105108,18.710,39.0000,45.49325,7.23,32.090,21031,21035,...,1052,28.433346,24.27,59.17930,0.380855,5256,39.024283,29.40,45.49325,0.522716
2,p227,S,105108,23.340,53.9200,64.50000,8.99,40.950,21026,21023,...,1052,29.882281,29.51,81.66930,0.400263,5258,42.575812,31.67,64.50000,0.570288
3,p227,ST,105108,15.210,33.2400,38.31000,5.41,26.918,21038,21022,...,1052,53.060418,29.37,46.36465,0.710725,5259,55.921709,29.31,38.31000,0.749051
4,p235,G,105108,24.280,54.0830,67.46650,10.74,41.200,21031,21023,...,1053,29.788072,29.45,90.07000,0.399001,5256,42.927430,30.70,67.46650,0.574998
5,p235,T,105108,12.830,34.0530,42.40000,4.59,25.680,21025,21026,...,1052,28.268289,24.28,57.67930,0.378644,5257,32.126823,29.33,42.40000,0.430327
6,p235,S,105108,20.100,53.4215,63.67000,8.05,39.096,21031,21022,...,1052,30.128479,29.51,80.72930,0.403560,5257,42.863688,31.63,63.67000,0.574144
7,p235,ST,105108,10.085,27.3715,32.33000,3.73,20.720,21036,21023,...,1052,32.132367,29.22,40.94930,0.430402,5259,35.652310,29.20,32.33000,0.477550


Saved validation summary to: flow_baseline_validation_2019\flow_leakage_validation_k4_main.csv


## Pressure-residual assessment

Compute 2019 pressure residuals and pressure-deficit scores under the four baseline variants.

In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

PRESSURE_FILE = Path("2019_SCADA_Pressures.csv")
LABEL_FILE = Path("pump_kmeans_screening_2019/pump_kmeans_labels_k2_12.csv")

OUTPUT_DIR = Path("pressure_baselines_2019")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP_COL = "Timestamp"
STATE_COL = "pump_state_k4"

PRESSURE_SIGNALS = [
    "n54", "n429", "n410", "n332", "n415",
    "n722", "n679", "n495", "n740"
]

BASELINES = ["G", "T", "S", "ST"]

SEP = ";"
TOD_NEIGHBOUR_SLOTS = 6     # +/- 30 min on 5-min grid
SLOTS_PER_DAY = 288         # 24 * 60 / 5


# Robust semicolon CSV reader
def read_semicolon_csv_robust(path: Path, timestamp_col: str = "Timestamp") -> pd.DataFrame:

    df = pd.read_csv(path, sep=SEP, dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found in {path}. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    if df[timestamp_col].isna().any():
        n_bad = int(df[timestamp_col].isna().sum())
        raise ValueError(f"Timestamp parsing failed for {n_bad} rows in {path}")

    for col in df.columns:
        if col == timestamp_col:
            continue

        s = df[col].astype(str).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
        s = s.str.replace(",", ".", regex=False)

        converted = pd.to_numeric(s, errors="coerce")

        if converted.notna().any():
            df[col] = converted
        else:
            df[col] = s

    df = df.sort_values(timestamp_col).reset_index(drop=True)
    return df


# Cyclic time-of-day baseline
# For each 5-min slot s, compute the median over slots in
# [s-k, ..., s, ..., s+k] modulo 288.
def compute_tod_median_baseline(
    values: pd.Series,
    slot_index: pd.Series,
    neighbour_slots: int = 6
) -> pd.Series:
    by_slot = {
        s: values[slot_index == s].dropna().to_numpy()
        for s in range(SLOTS_PER_DAY)
    }

    slot_medians = {}

    for s in range(SLOTS_PER_DAY):
        neighbour_values = []

        for off in range(-neighbour_slots, neighbour_slots + 1):
            s2 = (s + off) % SLOTS_PER_DAY
            arr = by_slot[s2]

            if arr.size:
                neighbour_values.append(arr)

        if neighbour_values:
            slot_medians[s] = float(np.median(np.concatenate(neighbour_values)))
        else:
            slot_medians[s] = np.nan

    return slot_index.map(slot_medians).astype(float)


# Pump-state + time-of-day baseline
# Same cyclic TOD logic, but only within the current pump state.
def compute_state_tod_median_baseline(
    values: pd.Series,
    state_labels: pd.Series,
    slot_index: pd.Series,
    neighbour_slots: int = 6
) -> pd.Series:
    unique_states = sorted([x for x in state_labels.dropna().unique()])
    slot_state_medians = {}

    for st in unique_states:
        mask_st = state_labels == st
        values_st = values[mask_st]
        slots_st = slot_index[mask_st]

        by_slot = {
            s: values_st[slots_st == s].dropna().to_numpy()
            for s in range(SLOTS_PER_DAY)
        }

        for s in range(SLOTS_PER_DAY):
            neighbour_values = []

            for off in range(-neighbour_slots, neighbour_slots + 1):
                s2 = (s + off) % SLOTS_PER_DAY
                arr = by_slot[s2]

                if arr.size:
                    neighbour_values.append(arr)

            if neighbour_values:
                slot_state_medians[(st, s)] = float(np.median(np.concatenate(neighbour_values)))
            else:
                slot_state_medians[(st, s)] = np.nan

    keys = list(zip(state_labels, slot_index))
    return pd.Series(
        [slot_state_medians.get(k, np.nan) for k in keys],
        index=values.index,
        dtype=float
    )


# Load and prepare data
pressures = read_semicolon_csv_robust(PRESSURE_FILE, timestamp_col=TIMESTAMP_COL)
labels = read_semicolon_csv_robust(LABEL_FILE, timestamp_col=TIMESTAMP_COL)

if STATE_COL not in labels.columns:
    raise ValueError(
        f"Required state column '{STATE_COL}' not found in {LABEL_FILE}. "
        f"Available columns: {list(labels.columns)}"
    )

missing_pressure_cols = [c for c in PRESSURE_SIGNALS if c not in pressures.columns]
if missing_pressure_cols:
    raise ValueError(
        f"Missing pressure columns in {PRESSURE_FILE}: {missing_pressure_cols}. "
        f"Available columns: {list(pressures.columns)}"
    )

pressures_use = pressures[[TIMESTAMP_COL] + PRESSURE_SIGNALS].copy()
labels_use = labels[[TIMESTAMP_COL, STATE_COL]].copy()

df = pressures_use.merge(labels_use, on=TIMESTAMP_COL, how="inner")
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

df["tod_slot_5min"] = (
    (df[TIMESTAMP_COL].dt.hour * 60 + df[TIMESTAMP_COL].dt.minute) // 5
).astype(int)

if df[STATE_COL].isna().any():
    raise ValueError(f"State column '{STATE_COL}' contains missing values after merge.")

print(f"Merged rows: {len(df):,}")
display(df[[TIMESTAMP_COL, STATE_COL] + PRESSURE_SIGNALS].head())


# Compute baselines, residuals, deficits, and absolute residuals
new_cols = {}
summary_rows = []

for pressure_col in PRESSURE_SIGNALS:
    p = df[pressure_col].astype(float)

    # --- Baselines
    baseline_G = pd.Series(float(p.median()), index=df.index, dtype=float)

    baseline_T = compute_tod_median_baseline(
        values=p,
        slot_index=df["tod_slot_5min"],
        neighbour_slots=TOD_NEIGHBOUR_SLOTS,
    )

    state_medians = p.groupby(df[STATE_COL]).median().to_dict()
    baseline_S = df[STATE_COL].map(state_medians).astype(float)

    baseline_ST = compute_state_tod_median_baseline(
        values=p,
        state_labels=df[STATE_COL],
        slot_index=df["tod_slot_5min"],
        neighbour_slots=TOD_NEIGHBOUR_SLOTS,
    )

    baseline_series = {
        "G": baseline_G,
        "T": baseline_T,
        "S": baseline_S,
        "ST": baseline_ST,
    }

    for b in BASELINES:
        base_col = f"{pressure_col}_baseline_{b}"
        resid_col = f"{pressure_col}_resid_{b}"
        deficit_col = f"{pressure_col}_deficit_{b}"
        absresid_col = f"{pressure_col}_absresid_{b}"

        baseline = baseline_series[b]
        residual = p - baseline
        deficit = np.maximum(0, -residual)
        absresid = residual.abs()

        new_cols[base_col] = baseline
        new_cols[resid_col] = residual
        new_cols[deficit_col] = deficit
        new_cols[absresid_col] = absresid

        valid = pd.DataFrame({
            "residual": residual,
            "deficit": deficit,
            "absresid": absresid,
        }).dropna()

        summary_rows.append({
            "pressure_signal": pressure_col,
            "baseline": b,
            "n_rows": int(len(valid)),
            "median_residual": float(valid["residual"].median()) if len(valid) else np.nan,
            "mean_residual": float(valid["residual"].mean()) if len(valid) else np.nan,
            "share_negative_residual": float((valid["residual"] < 0).mean()) if len(valid) else np.nan,
            "median_deficit": float(valid["deficit"].median()) if len(valid) else np.nan,
            "mean_deficit": float(valid["deficit"].mean()) if len(valid) else np.nan,
            "p90_deficit": float(valid["deficit"].quantile(0.90)) if len(valid) else np.nan,
            "p95_deficit": float(valid["deficit"].quantile(0.95)) if len(valid) else np.nan,
            "median_absresid": float(valid["absresid"].median()) if len(valid) else np.nan,
            "mean_absresid": float(valid["absresid"].mean()) if len(valid) else np.nan,
            "p90_absresid": float(valid["absresid"].quantile(0.90)) if len(valid) else np.nan,
            "p95_absresid": float(valid["absresid"].quantile(0.95)) if len(valid) else np.nan,
        })

# Add all calculated columns at once to avoid dataframe-fragmentation warnings
df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

summary_df = pd.DataFrame(summary_rows)


# Add relative reduction against global baseline
# Negative values mean the baseline produced larger residuals than G.
for metric in ["median_absresid", "mean_absresid", "p90_absresid", "p95_absresid",
               "median_deficit", "mean_deficit", "p90_deficit", "p95_deficit"]:
    g_values = (
        summary_df.loc[summary_df["baseline"] == "G", ["pressure_signal", metric]]
        .rename(columns={metric: f"{metric}_G"})
    )

    summary_df = summary_df.merge(g_values, on="pressure_signal", how="left")

    summary_df[f"relative_reduction_vs_G_{metric}_pct"] = np.where(
        summary_df[f"{metric}_G"].notna() & (summary_df[f"{metric}_G"] != 0),
        100.0 * (summary_df[f"{metric}_G"] - summary_df[metric]) / summary_df[f"{metric}_G"],
        np.nan
    )

    summary_df = summary_df.drop(columns=[f"{metric}_G"])


# Display compact summaries
display(summary_df)

print("\nMedian pressure-deficit summary:")
display(
    summary_df.pivot(
        index="pressure_signal",
        columns="baseline",
        values="median_deficit"
    )
)

print("\nP95 pressure-deficit summary:")
display(
    summary_df.pivot(
        index="pressure_signal",
        columns="baseline",
        values="p95_deficit"
    )
)


timestamp_output = OUTPUT_DIR / "pressure_timestamp_baselines_and_residuals_k4_main.csv"
summary_output = OUTPUT_DIR / "pressure_baseline_residual_summary_k4_main.csv"

df.to_csv(timestamp_output, sep=";", index=False)
summary_df.to_csv(summary_output, sep=";", index=False)


Merged rows: 105,108


,Timestamp,pump_state_k4,n54,n429,n410,n332,n415,n722,n679,n495,n740
0,2019-01-01 01:00:00,0,37.22,36.88,31.16,56.51,45.71,46.17,47.37,51.75,43.82
1,2019-01-01 01:05:00,0,37.27,36.93,31.21,56.57,45.77,46.21,47.41,51.78,43.84
2,2019-01-01 01:10:00,0,37.26,36.92,31.21,56.55,45.75,46.20,47.41,51.77,43.84
3,2019-01-01 01:15:00,0,37.32,36.98,31.27,56.59,45.80,46.23,47.44,51.79,43.85
4,2019-01-01 01:20:00,0,37.32,36.98,31.27,56.58,45.80,46.22,47.44,51.79,43.84


,pressure_signal,baseline,n_rows,median_residual,mean_residual,share_negative_residual,median_deficit,mean_deficit,p90_deficit,p95_deficit,...,p90_absresid,p95_absresid,relative_reduction_vs_G_median_absresid_pct,relative_reduction_vs_G_mean_absresid_pct,relative_reduction_vs_G_p90_absresid_pct,relative_reduction_vs_G_p95_absresid_pct,relative_reduction_vs_G_median_deficit_pct,relative_reduction_vs_G_mean_deficit_pct,relative_reduction_vs_G_p90_deficit_pct,relative_reduction_vs_G_p95_deficit_pct
0,n54,G,105108,0.0,-0.010062,0.498097,0.0,0.436791,1.30,1.49,...,1.550,1.76,0.000000,0.000000,0.000000,0.000000e+00,NaN,0.000000,0.000000,0.000000
1,n54,T,105108,0.0,0.146624,0.499049,0.0,0.237262,0.73,0.89,...,1.270,1.64,37.647059,28.067856,18.064516,6.818182e+00,NaN,45.680660,43.846154,40.268456
2,n54,S,105108,0.0,0.087478,0.496251,0.0,0.296231,0.93,1.12,...,1.330,1.55,25.882353,21.259612,14.193548,1.193182e+01,NaN,32.180301,28.461538,24.832215
3,n54,ST,105108,0.0,0.039027,0.496946,0.0,0.184163,0.58,0.72,...,0.870,1.05,60.000000,52.826447,43.870968,4.034091e+01,NaN,57.837300,55.384615,51.677852
4,n429,G,105108,0.0,-0.018848,0.498563,0.0,0.428125,1.28,1.48,...,1.510,1.70,0.000000,0.000000,0.000000,0.000000e+00,NaN,0.000000,0.000000,0.000000
5,n429,T,105108,0.0,0.121259,0.499305,0.0,0.236586,0.73,0.89,...,1.190,1.53,37.804878,29.014631,21.192053,1.000000e+01,NaN,44.738888,42.968750,39.864865
6,n429,S,105108,0.0,0.079865,0.498221,0.0,0.300454,0.94,1.13,...,1.330,1.55,23.170732,18.704063,11.920530,8.823529e+00,NaN,29.820844,26.562500,23.648649
7,n429,ST,105108,0.0,0.036229,0.497231,0.0,0.187255,0.59,0.73,...,0.870,1.05,58.536585,50.950788,42.384106,3.823529e+01,NaN,56.261627,53.906250,50.675676
8,n410,G,105108,0.0,-0.032403,0.497707,0.0,0.470483,1.39,1.59,...,1.620,1.80,0.000000,0.000000,0.000000,0.000000e+00,NaN,0.000000,0.000000,0.000000
9,n410,T,105108,0.0,0.165018,0.499372,0.0,0.242395,0.74,0.91,...,1.310,1.69,38.888889,28.479626,19.135802,6.111111e+00,NaN,48.479571,46.762590,42.767296



Median pressure-deficit summary:


baseline,G,S,ST,T
pressure_signal,,,,
n332,0.0,0.0,0.0,0.0
n410,0.0,0.0,0.0,0.0
n415,0.0,0.0,0.0,0.0
n429,0.0,0.0,0.0,0.0
n495,0.0,0.0,0.0,0.0
n54,0.0,0.0,0.0,0.0
n679,0.0,0.0,0.0,0.0
n722,0.0,0.0,0.0,0.0
n740,0.0,0.0,0.0,0.0



P95 pressure-deficit summary:


baseline,G,S,ST,T
pressure_signal,,,,
n332,0.78,0.65,0.39,0.46
n410,1.59,1.22,0.77,0.91
n415,1.19,0.94,0.58,0.70
n429,1.48,1.13,0.73,0.89
n495,0.46,0.38,0.23,0.27
n54,1.49,1.12,0.72,0.89
n679,1.15,0.99,0.67,0.77
n722,0.90,0.79,0.55,0.62
n740,0.45,0.39,0.27,0.31



Saved timestamp-level output to: pressure_baselines_2019\pressure_timestamp_baselines_and_residuals_k4_main.csv
Saved summary output to:        pressure_baselines_2019\pressure_baseline_residual_summary_k4_main.csv


## Leakage-oriented validation of pressure deficits

Validate 2019 pressure-deficit scores against total leakage intensity and the high-leakage-intensity indicator.

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

PRESSURE_RESID_FILE = Path("pressure_baselines_2019/pressure_timestamp_baselines_and_residuals_k4_main.csv")
LEAK_FILE = Path("2019_Leakages.csv")

TIMESTAMP_COL = "Timestamp"
SEP = ";"

PRESSURE_SIGNALS = [
    "n54", "n429", "n410", "n332", "n415",
    "n722", "n679", "n495", "n740"
]

BASELINES = ["G", "T", "S", "ST"]

LOW_Q = 0.20
HIGH_Q = 0.80
TOP_PCTS = [0.01, 0.05]
HIGH_LEAK_QUANTILE = 0.80

OUTPUT_DIR = Path("pressure_baseline_validation_2019")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Robust semicolon CSV reader
def read_semicolon_csv_robust(path: Path, timestamp_col: str = "Timestamp") -> pd.DataFrame:

    df = pd.read_csv(path, sep=SEP, dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if timestamp_col not in df.columns:
        raise ValueError(
            f"Timestamp column '{timestamp_col}' not found in {path}. "
            f"Available columns: {list(df.columns)}"
        )

    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    if df[timestamp_col].isna().any():
        n_bad = int(df[timestamp_col].isna().sum())
        raise ValueError(f"Timestamp parsing failed for {n_bad} rows in {path}")

    for col in df.columns:
        if col == timestamp_col:
            continue

        s = df[col].astype(str).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
        s = s.str.replace(",", ".", regex=False)

        converted = pd.to_numeric(s, errors="coerce")

        if converted.notna().any():
            df[col] = converted
        else:
            df[col] = s

    df = df.sort_values(timestamp_col).reset_index(drop=True)
    return df


def safe_mean(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.mean()) if len(x) else np.nan


def safe_median(x: pd.Series) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.median()) if len(x) else np.nan


def safe_quantile(x: pd.Series, q: float) -> float:
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.quantile(q)) if len(x) else np.nan


def low_high_score_contrast(
    df_in: pd.DataFrame,
    score_col: str,
    leak_flow_col: str = "total_leak_flow",
    high_leak_col: str = "high_leak_intensity",
    low_q: float = 0.20,
    high_q: float = 0.80
) -> dict:
    df_work = df_in[[score_col, leak_flow_col, high_leak_col]].dropna().copy()

    if df_work.empty:
        return {
            "low_thr": np.nan,
            "high_thr": np.nan,
            "low_n": 0,
            "high_n": 0,
            "low_total_leak_mean": np.nan,
            "high_total_leak_mean": np.nan,
            "low_total_leak_median": np.nan,
            "high_total_leak_median": np.nan,
            "high_low_total_leak_mean_ratio": np.nan,
            "low_high_leak_share": np.nan,
            "high_high_leak_share": np.nan,
            "high_low_high_leak_share_ratio": np.nan,
        }

    low_thr = df_work[score_col].quantile(low_q)
    high_thr = df_work[score_col].quantile(high_q)

    low_df = df_work.loc[df_work[score_col] <= low_thr]
    high_df = df_work.loc[df_work[score_col] >= high_thr]

    low_leak_mean = safe_mean(low_df[leak_flow_col])
    high_leak_mean = safe_mean(high_df[leak_flow_col])

    low_high_leak_share = safe_mean(low_df[high_leak_col])
    high_high_leak_share = safe_mean(high_df[high_leak_col])

    return {
        "low_thr": float(low_thr),
        "high_thr": float(high_thr),
        "low_n": int(len(low_df)),
        "high_n": int(len(high_df)),
        "low_total_leak_mean": low_leak_mean,
        "high_total_leak_mean": high_leak_mean,
        "low_total_leak_median": safe_median(low_df[leak_flow_col]),
        "high_total_leak_median": safe_median(high_df[leak_flow_col]),
        "high_low_total_leak_mean_ratio": (
            float(high_leak_mean / low_leak_mean)
            if pd.notna(low_leak_mean) and low_leak_mean != 0
            else np.nan
        ),
        "low_high_leak_share": low_high_leak_share,
        "high_high_leak_share": high_high_leak_share,
        "high_low_high_leak_share_ratio": (
            float(high_high_leak_share / low_high_leak_share)
            if pd.notna(low_high_leak_share) and low_high_leak_share != 0
            else np.nan
        ),
    }


def top_tail_enrichment(
    df_in: pd.DataFrame,
    score_col: str,
    top_pct: float,
    leak_flow_col: str = "total_leak_flow",
    high_leak_col: str = "high_leak_intensity"
) -> dict:
    df_work = df_in[[score_col, leak_flow_col, high_leak_col]].dropna().copy()
    tag = f"top_{int(top_pct * 100)}pct"

    if df_work.empty:
        return {
            f"{tag}_n": 0,
            f"{tag}_threshold": np.nan,
            f"{tag}_total_leak_mean": np.nan,
            f"{tag}_total_leak_median": np.nan,
            f"{tag}_total_leak_enrichment_ratio": np.nan,
            f"{tag}_high_leak_share": np.nan,
            f"{tag}_high_leak_enrichment_ratio": np.nan,
        }

    thr = df_work[score_col].quantile(1 - top_pct)
    top_df = df_work.loc[df_work[score_col] >= thr]

    global_leak_mean = safe_mean(df_work[leak_flow_col])
    top_leak_mean = safe_mean(top_df[leak_flow_col])

    global_high_leak_share = safe_mean(df_work[high_leak_col])
    top_high_leak_share = safe_mean(top_df[high_leak_col])

    return {
        f"{tag}_n": int(len(top_df)),
        f"{tag}_threshold": float(thr),
        f"{tag}_total_leak_mean": top_leak_mean,
        f"{tag}_total_leak_median": safe_median(top_df[leak_flow_col]),
        f"{tag}_total_leak_enrichment_ratio": (
            float(top_leak_mean / global_leak_mean)
            if pd.notna(global_leak_mean) and global_leak_mean != 0
            else np.nan
        ),
        f"{tag}_high_leak_share": top_high_leak_share,
        f"{tag}_high_leak_enrichment_ratio": (
            float(top_high_leak_share / global_high_leak_share)
            if pd.notna(global_high_leak_share) and global_high_leak_share != 0
            else np.nan
        ),
    }


def high_leak_score_separation(
    df_in: pd.DataFrame,
    score_col: str,
    high_leak_col: str = "high_leak_intensity"
) -> dict:
    df_work = df_in[[score_col, high_leak_col]].dropna().copy()

    if df_work.empty:
        return {
            "nohigh_n": 0,
            "highleak_n": 0,
            "score_mean_nohigh": np.nan,
            "score_mean_highleak": np.nan,
            "score_median_nohigh": np.nan,
            "score_median_highleak": np.nan,
            "score_p90_nohigh": np.nan,
            "score_p90_highleak": np.nan,
            "score_p95_nohigh": np.nan,
            "score_p95_highleak": np.nan,
            "score_mean_high_minus_nohigh": np.nan,
            "score_median_high_minus_nohigh": np.nan,
            "score_p90_high_minus_nohigh": np.nan,
            "score_p95_high_minus_nohigh": np.nan,
            "score_mean_high_nohigh_ratio": np.nan,
        }

    nohigh = df_work.loc[df_work[high_leak_col] == 0, score_col]
    high = df_work.loc[df_work[high_leak_col] == 1, score_col]

    mean_nohigh = safe_mean(nohigh)
    mean_high = safe_mean(high)

    median_nohigh = safe_median(nohigh)
    median_high = safe_median(high)

    p90_nohigh = safe_quantile(nohigh, 0.90)
    p90_high = safe_quantile(high, 0.90)

    p95_nohigh = safe_quantile(nohigh, 0.95)
    p95_high = safe_quantile(high, 0.95)

    return {
        "nohigh_n": int(nohigh.notna().sum()),
        "highleak_n": int(high.notna().sum()),
        "score_mean_nohigh": mean_nohigh,
        "score_mean_highleak": mean_high,
        "score_median_nohigh": median_nohigh,
        "score_median_highleak": median_high,
        "score_p90_nohigh": p90_nohigh,
        "score_p90_highleak": p90_high,
        "score_p95_nohigh": p95_nohigh,
        "score_p95_highleak": p95_high,
        "score_mean_high_minus_nohigh": (
            float(mean_high - mean_nohigh)
            if pd.notna(mean_high) and pd.notna(mean_nohigh)
            else np.nan
        ),
        "score_median_high_minus_nohigh": (
            float(median_high - median_nohigh)
            if pd.notna(median_high) and pd.notna(median_nohigh)
            else np.nan
        ),
        "score_p90_high_minus_nohigh": (
            float(p90_high - p90_nohigh)
            if pd.notna(p90_high) and pd.notna(p90_nohigh)
            else np.nan
        ),
        "score_p95_high_minus_nohigh": (
            float(p95_high - p95_nohigh)
            if pd.notna(p95_high) and pd.notna(p95_nohigh)
            else np.nan
        ),
        "score_mean_high_nohigh_ratio": (
            float(mean_high / mean_nohigh)
            if pd.notna(mean_high) and pd.notna(mean_nohigh) and mean_nohigh != 0
            else np.nan
        ),
    }


# Load pressure residuals and leakage data
pressure = read_semicolon_csv_robust(PRESSURE_RESID_FILE, timestamp_col=TIMESTAMP_COL)
leak = read_semicolon_csv_robust(LEAK_FILE, timestamp_col=TIMESTAMP_COL)

# Derive aggregate leakage variables from 2019_Leakages.csv
leak_cols = [c for c in leak.columns if c != TIMESTAMP_COL]

if not leak_cols:
    raise ValueError("No leakage columns found in leakage file.")

leak["total_leak_flow"] = leak[leak_cols].sum(axis=1)
leak["leak_active"] = (leak["total_leak_flow"] > 0).astype(int)
leak["active_leak_count"] = (leak[leak_cols] > 0).sum(axis=1)
leak["max_leak_flow"] = leak[leak_cols].max(axis=1)

nonzero_total = leak.loc[leak["total_leak_flow"] > 0, "total_leak_flow"]

if len(nonzero_total):
    high_thr_global = float(nonzero_total.quantile(HIGH_LEAK_QUANTILE))
else:
    high_thr_global = 0.0

leak["high_leak_intensity"] = (
    (leak["total_leak_flow"] >= high_thr_global) &
    (leak["total_leak_flow"] > 0)
).astype(int)

leak_use = leak[
    [
        TIMESTAMP_COL,
        "total_leak_flow",
        "leak_active",
        "active_leak_count",
        "max_leak_flow",
        "high_leak_intensity",
    ]
].copy()

print(f"High-leak threshold, nonzero total leak q={HIGH_LEAK_QUANTILE:.2f}: {high_thr_global:.6f}")
display(leak_use.head())


# Merge pressure scores with leakage context
df = pressure.merge(leak_use, on=TIMESTAMP_COL, how="inner")
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

print(f"Merged rows for pressure validation: {len(df):,}")
display(df[[TIMESTAMP_COL, "total_leak_flow", "high_leak_intensity"]].head())


# Validate pressure-deficit scores against leakage labels
validation_rows = []
separation_rows = []

for pressure_col in PRESSURE_SIGNALS:
    for b in BASELINES:
        score_col = f"{pressure_col}_deficit_{b}"

        if score_col not in df.columns:
            raise ValueError(f"Missing pressure-deficit score column: {score_col}")

        score = pd.to_numeric(df[score_col], errors="coerce")

        row = {
            "pressure_signal": pressure_col,
            "baseline": b,
            "score_col": score_col,
            "n_rows": int(score.notna().sum()),
            "score_mean": safe_mean(score),
            "score_median": safe_median(score),
            "score_p90": safe_quantile(score, 0.90),
            "score_p95": safe_quantile(score, 0.95),
            "score_p99": safe_quantile(score, 0.99),
            "share_positive_deficit": float((score > 0).mean()),
        }

        row.update(
            low_high_score_contrast(
                df_in=df,
                score_col=score_col,
                leak_flow_col="total_leak_flow",
                high_leak_col="high_leak_intensity",
                low_q=LOW_Q,
                high_q=HIGH_Q,
            )
        )

        for top_pct in TOP_PCTS:
            row.update(
                top_tail_enrichment(
                    df_in=df,
                    score_col=score_col,
                    top_pct=top_pct,
                    leak_flow_col="total_leak_flow",
                    high_leak_col="high_leak_intensity",
                )
            )

        validation_rows.append(row)

        sep_row = {
            "pressure_signal": pressure_col,
            "baseline": b,
            "score_col": score_col,
        }
        sep_row.update(
            high_leak_score_separation(
                df_in=df,
                score_col=score_col,
                high_leak_col="high_leak_intensity",
            )
        )
        separation_rows.append(sep_row)

validation = pd.DataFrame(validation_rows)
separation = pd.DataFrame(separation_rows)

baseline_order = pd.Categorical(validation["baseline"], categories=BASELINES, ordered=True)
validation = (
    validation.assign(_baseline_order=baseline_order)
    .sort_values(["pressure_signal", "_baseline_order"])
    .drop(columns="_baseline_order")
    .reset_index(drop=True)
)

baseline_order_sep = pd.Categorical(separation["baseline"], categories=BASELINES, ordered=True)
separation = (
    separation.assign(_baseline_order=baseline_order_sep)
    .sort_values(["pressure_signal", "_baseline_order"])
    .drop(columns="_baseline_order")
    .reset_index(drop=True)
)

display(validation)
display(separation)


# Compact baseline-ranking table
# Higher values indicate stronger leakage-oriented enrichment/separation.
ranking_rows = []

for pressure_col in PRESSURE_SIGNALS:
    sub = validation.loc[validation["pressure_signal"] == pressure_col].copy()

    for metric in [
        "top_1pct_high_leak_enrichment_ratio",
        "top_5pct_high_leak_enrichment_ratio",
        "top_1pct_total_leak_enrichment_ratio",
        "top_5pct_total_leak_enrichment_ratio",
        "high_low_total_leak_mean_ratio",
        "high_low_high_leak_share_ratio",
    ]:
        if metric not in sub.columns:
            continue

        valid_metric = sub.dropna(subset=[metric])
        if valid_metric.empty:
            best_baseline = np.nan
            best_value = np.nan
        else:
            best_idx = valid_metric[metric].idxmax()
            best_baseline = valid_metric.loc[best_idx, "baseline"]
            best_value = valid_metric.loc[best_idx, metric]

        ranking_rows.append({
            "pressure_signal": pressure_col,
            "metric": metric,
            "best_baseline": best_baseline,
            "best_value": best_value,
        })

ranking = pd.DataFrame(ranking_rows)

display(ranking)


print("\nTop 1% high-leak enrichment ratio:")
display(
    validation.pivot(
        index="pressure_signal",
        columns="baseline",
        values="top_1pct_high_leak_enrichment_ratio"
    )
)

print("\nTop 5% high-leak enrichment ratio:")
display(
    validation.pivot(
        index="pressure_signal",
        columns="baseline",
        values="top_5pct_high_leak_enrichment_ratio"
    )
)

print("\nHigh-score vs low-score total leak mean ratio:")
display(
    validation.pivot(
        index="pressure_signal",
        columns="baseline",
        values="high_low_total_leak_mean_ratio"
    )
)

print("\nHigh-leak vs no/low-leak mean pressure-deficit ratio:")
display(
    separation.pivot(
        index="pressure_signal",
        columns="baseline",
        values="score_mean_high_nohigh_ratio"
    )
)


validation_out = OUTPUT_DIR / "pressure_leakage_validation_k4_main.csv"
separation_out = OUTPUT_DIR / "pressure_high_leak_score_separation_k4_main.csv"
ranking_out = OUTPUT_DIR / "pressure_baseline_ranking_k4_main.csv"
merged_out = OUTPUT_DIR / "pressure_timestamp_scores_with_leakage_k4_main.csv"

validation.to_csv(validation_out, sep=";", index=False)
separation.to_csv(separation_out, sep=";", index=False)
ranking.to_csv(ranking_out, sep=";", index=False)
df.to_csv(merged_out, sep=";", index=False)


High-leak threshold, nonzero total leak q=0.80: 103.670000


,Timestamp,total_leak_flow,leak_active,active_leak_count,max_leak_flow,high_leak_intensity
0,2019-01-01 00:00:00,24.16,1,4,6.87,0
1,2019-01-01 00:05:00,24.18,1,4,6.87,0
2,2019-01-01 00:10:00,24.18,1,4,6.87,0
3,2019-01-01 00:15:00,24.18,1,4,6.87,0
4,2019-01-01 00:20:00,24.17,1,4,6.87,0


Merged rows for pressure validation: 105,108


,Timestamp,total_leak_flow,high_leak_intensity
0,2019-01-01 01:00:00,24.23,0
1,2019-01-01 01:05:00,24.23,0
2,2019-01-01 01:10:00,24.23,0
3,2019-01-01 01:15:00,24.24,0
4,2019-01-01 01:20:00,24.24,0


,pressure_signal,baseline,score_col,n_rows,score_mean,score_median,score_p90,score_p95,score_p99,share_positive_deficit,...,top_1pct_total_leak_enrichment_ratio,top_1pct_high_leak_share,top_1pct_high_leak_enrichment_ratio,top_5pct_n,top_5pct_threshold,top_5pct_total_leak_mean,top_5pct_total_leak_median,top_5pct_total_leak_enrichment_ratio,top_5pct_high_leak_share,top_5pct_high_leak_enrichment_ratio
0,n332,G,n332_deficit_G,105108,0.220057,0.0,0.66,0.78,0.98,0.497089,...,1.388925,0.585887,2.928403,5359,0.78,102.108608,105.630,1.367708,0.525471,2.626431
1,n332,T,n332_deficit_T,105108,0.115663,0.0,0.36,0.46,0.65,0.494625,...,1.395265,0.587199,2.934959,5319,0.46,103.544213,107.990,1.386938,0.553111,2.764584
2,n332,S,n332_deficit_S,105108,0.175002,0.0,0.54,0.65,0.86,0.498839,...,1.373186,0.563380,2.815910,5259,0.65,99.792333,102.240,1.336683,0.485073,2.424513
3,n332,ST,n332_deficit_ST,105108,0.096957,0.0,0.31,0.39,0.55,0.494434,...,1.411640,0.619349,3.095655,5424,0.39,103.930383,110.950,1.392110,0.567847,2.838234
4,n410,G,n410_deficit_G,105108,0.470483,0.0,1.39,1.59,1.91,0.497707,...,1.475473,0.697026,3.483904,5468,1.59,108.220501,112.235,1.449575,0.630212,3.149952
5,n410,T,n410_deficit_T,105108,0.242395,0.0,0.74,0.91,1.29,0.499372,...,1.481717,0.692015,3.458858,5489,0.91,108.178249,111.890,1.449009,0.617781,3.087818
6,n410,S,n410_deficit_S,105108,0.328688,0.0,1.02,1.22,1.57,0.498659,...,1.466005,0.686623,3.431907,5361,1.22,106.236877,109.720,1.423005,0.586271,2.930325
7,n410,ST,n410_deficit_ST,105108,0.197137,0.0,0.61,0.77,1.12,0.497764,...,1.476166,0.670381,3.350724,5363,0.77,107.621926,113.400,1.441557,0.623532,3.116561
8,n415,G,n415_deficit_G,105108,0.341769,0.0,1.02,1.19,1.46,0.496984,...,1.417742,0.610427,3.051059,5372,1.19,105.447969,108.970,1.412438,0.578183,2.889899
9,n415,T,n415_deficit_T,105108,0.181791,0.0,0.56,0.70,0.98,0.497336,...,1.432409,0.622873,3.113271,5295,0.70,106.284761,109.790,1.423647,0.587535,2.936643


...


High-score vs low-score total leak mean ratio:


baseline,G,S,ST,T
pressure_signal,,,,
n332,1.591228,1.541432,1.844183,1.847588
n410,1.696817,1.664916,2.022402,1.978507
n415,1.640316,1.618181,1.982516,1.939527
n429,1.700461,1.692568,2.080486,2.003788
n495,1.592222,1.536659,1.850135,1.852496
n54,1.682655,1.682901,2.064850,1.977507
n679,1.802396,1.796570,2.215716,2.201085
n722,1.806458,1.786714,2.195859,2.194641
n740,1.801629,1.784529,2.196065,2.193918



High-leak vs no/low-leak mean pressure-deficit ratio:


baseline,G,S,ST,T
pressure_signal,,,,
n332,2.200993,2.172111,3.524443,3.391981
n410,2.394984,2.596999,4.335199,3.870631
n415,2.322878,2.471893,4.119142,3.697030
n429,2.498878,2.792692,4.818784,4.079958
n495,2.207586,2.211028,3.620774,3.472489
n54,2.426757,2.750759,4.697959,3.926383
n679,3.107713,3.314140,5.956599,5.334122
n722,3.027081,3.158340,5.258810,4.915852
n740,3.056464,3.218738,5.438917,5.038945



Saved pressure validation summary to:   pressure_baseline_validation_2019\pressure_leakage_validation_k4_main.csv
Saved high-leak separation summary to: pressure_baseline_validation_2019\pressure_high_leak_score_separation_k4_main.csv
Saved baseline ranking summary to:     pressure_baseline_validation_2019\pressure_baseline_ranking_k4_main.csv
Saved merged timestamp-level file to:  pressure_baseline_validation_2019\pressure_timestamp_scores_with_leakage_k4_main.csv


## Cross-year flow-tail diagnostic

Compare the 2018 and 2019 flow-validation behaviour and the leakage composition of high-score flow-deviation tails.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

SEP = ";"
TIMESTAMP_COL = "Timestamp"
YEARS = [2018, 2019]
FLOW_SIGNALS = ["p235", "p227"]
OUT_DIR = Path("year_comparison_diagnostics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEAKAGE_FILES = {
    2018: Path("2018_Leakages.csv"),
    2019: Path("2019_Leakages.csv"),
}


def read_semicolon_csv(path: Path, timestamp_col: str = TIMESTAMP_COL) -> pd.DataFrame:
    df = pd.read_csv(path, sep=SEP, dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")

    for col in df.columns:
        if col == timestamp_col:
            continue
        s = df[col].astype(str).str.strip().str.replace(",", ".", regex=False)
        df[col] = pd.to_numeric(s, errors="coerce")

    return df.sort_values(timestamp_col).reset_index(drop=True)


leakage_year_rows = []
pipe_contrib_rows = []

for year in YEARS:
    leak = read_semicolon_csv(LEAKAGE_FILES[year])
    leak_cols = [c for c in leak.columns if c != TIMESTAMP_COL]

    leak["total_leak_flow"] = leak[leak_cols].sum(axis=1)
    leak["active_leak_count"] = (leak[leak_cols] > 0).sum(axis=1)
    nonzero_total = leak.loc[leak["total_leak_flow"] > 0, "total_leak_flow"]
    high_leak_threshold = nonzero_total.quantile(0.80)

    total_volume = float(leak["total_leak_flow"].sum())

    leakage_year_rows.append({
        "year": year,
        "n_rows": int(len(leak)),
        "n_leak_columns": int(len(leak_cols)),
        "total_leak_sum": total_volume,
        "mean_total_leak_flow": float(leak["total_leak_flow"].mean()),
        "median_total_leak_flow": float(leak["total_leak_flow"].median()),
        "p90_total_leak_flow": float(leak["total_leak_flow"].quantile(0.90)),
        "p95_total_leak_flow": float(leak["total_leak_flow"].quantile(0.95)),
        "p99_total_leak_flow": float(leak["total_leak_flow"].quantile(0.99)),
        "active_leak_share": float((leak["total_leak_flow"] > 0).mean()),
        "high_leak_threshold_nonzero_q80": float(high_leak_threshold),
        "high_leak_share_all_timestamps": float((leak["total_leak_flow"] >= high_leak_threshold).mean()),
        "mean_active_leak_count": float(leak["active_leak_count"].mean()),
        "max_active_leak_count": int(leak["active_leak_count"].max()),
    })

    for pipe in leak_cols:
        pipe_sum = float(leak[pipe].sum())
        active = leak[pipe] > 0
        pipe_contrib_rows.append({
            "year": year,
            "pipe": pipe,
            "pipe_leak_sum": pipe_sum,
            "pipe_contribution_share": pipe_sum / total_volume if total_volume > 0 else np.nan,
            "pipe_active_share": float(active.mean()),
            "pipe_mean_when_active": float(leak.loc[active, pipe].mean()) if active.any() else 0.0,
            "pipe_max": float(leak[pipe].max()),
        })

leakage_year_summary = pd.DataFrame(leakage_year_rows)
pipe_contrib = pd.DataFrame(pipe_contrib_rows).sort_values(
    ["year", "pipe_contribution_share"],
    ascending=[True, False]
)

leakage_year_summary.to_csv(OUT_DIR / "leakage_year_summary_2018_vs_2019.csv", sep=SEP, index=False)
pipe_contrib.to_csv(OUT_DIR / "leakage_pipe_contribution_2018_vs_2019.csv", sep=SEP, index=False)

print("Leakage year summary")
display(leakage_year_summary)

print("Top 15 leaking pipes by contribution in each year")
display(pipe_contrib.groupby("year").head(15))

print("Selected local pipes p235 and p227, if present in leakage records")
display(pipe_contrib[pipe_contrib["pipe"].isin(FLOW_SIGNALS)].sort_values(["pipe", "year"]))


Leakage year summary


,year,n_rows,n_leak_columns,total_leak_sum,mean_total_leak_flow,median_total_leak_flow,p90_total_leak_flow,p95_total_leak_flow,p99_total_leak_flow,active_leak_share,high_leak_threshold_nonzero_q80,high_leak_share_all_timestamps,mean_active_leak_count,max_active_leak_count
0,2018,105120,14,2683089.92,25.524067,23.91,46.500,57.12,79.0900,0.978025,43.502,0.195605,3.447717,6
1,2019,105120,23,7847307.33,74.650945,75.91,119.541,132.01,138.5981,1.000000,103.670,0.200048,10.264098,16


Top 15 leaking pipes by contribution in each year


,year,pipe,pipe_leak_sum,pipe_contribution_share,pipe_active_share,pipe_mean_when_active,pipe_max
4,2018,p257,671477.22,0.250263,0.978025,6.531244,6.87
6,2018,p427,378743.45,0.141159,0.873487,4.124802,5.11
7,2018,p461,234940.82,0.087564,0.187652,11.910211,30.54
10,2018,p654,204503.75,0.076219,0.486625,3.997806,5.49
9,2018,p628,182138.66,0.067884,0.074268,23.330173,35.38
12,2018,p810,180768.91,0.067373,0.422527,4.069905,6.91
11,2018,p673,145446.50,0.054209,0.048716,28.401972,28.42
8,2018,p538,139173.57,0.051871,0.040820,32.433831,32.84
1,2018,p158,122922.27,0.045814,0.047841,24.442686,24.76
2,2018,p183,120114.17,0.044767,0.070167,16.284459,16.46


Selected local pipes p235 and p227, if present in leakage records


,year,pipe,pipe_leak_sum,pipe_contribution_share,pipe_active_share,pipe_mean_when_active,pipe_max


## Cross-year leakage-composition diagnostic

Prepare compact cross-year leakage and flow-tail diagnostic outputs used for the case-study interpretation.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

SEP = ";"
TIMESTAMP_COL = "Timestamp"
STATE_COL = "pump_state_k4"

YEARS = [2018, 2019]
FLOW_SIGNALS = ["p235", "p227"]
BASELINES = ["G", "T", "S", "ST"]
TOP_PCTS = [0.01, 0.05]

TOD_NEIGHBOUR_SLOTS = 6
SLOTS_PER_DAY = 288
HIGH_LEAK_QUANTILE = 0.80

OUT_DIR = Path("year_comparison_diagnostics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    2018: {
        "flows": Path("2018_SCADA_Flows.csv"),
        "leakages": Path("2018_Leakages.csv"),
        "states": Path("pump_kmeans_screening_2018/pump_kmeans_labels_k2_12.csv"),
    },
    2019: {
        "flows": Path("2019_SCADA_Flows.csv"),
        "leakages": Path("2019_Leakages.csv"),
        "states": Path("pump_kmeans_screening_2019/pump_kmeans_labels_k2_12.csv"),
    },
}


def read_semicolon_csv_robust(path: Path, timestamp_col: str = "Timestamp") -> pd.DataFrame:

    df = pd.read_csv(path, sep=SEP, dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if timestamp_col in df.columns:
        df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
        if df[timestamp_col].isna().any():
            n_bad = int(df[timestamp_col].isna().sum())
            raise ValueError(f"Timestamp parsing failed for {n_bad} rows in {path}")

    for col in df.columns:
        if col == timestamp_col:
            continue

        s = df[col].astype(str).str.strip()
        s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
        s = s.str.replace(",", ".", regex=False)

        converted = pd.to_numeric(s, errors="coerce")
        if converted.notna().any():
            df[col] = converted
        else:
            df[col] = s

    if timestamp_col in df.columns:
        df = df.sort_values(timestamp_col).reset_index(drop=True)

    return df


def safe_ratio(num, den):
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return float(num / den)


# Baseline helpers
def compute_tod_median_baseline(
    values: pd.Series,
    slot_index: pd.Series,
    neighbour_slots: int = 6
) -> pd.Series:
    by_slot = {
        s: values[slot_index == s].dropna().to_numpy()
        for s in range(SLOTS_PER_DAY)
    }

    slot_medians = {}

    for s in range(SLOTS_PER_DAY):
        neighbour_values = []

        for off in range(-neighbour_slots, neighbour_slots + 1):
            s2 = (s + off) % SLOTS_PER_DAY
            arr = by_slot[s2]
            if arr.size:
                neighbour_values.append(arr)

        if neighbour_values:
            slot_medians[s] = float(np.median(np.concatenate(neighbour_values)))
        else:
            slot_medians[s] = np.nan

    return slot_index.map(slot_medians).astype(float)


def compute_state_tod_median_baseline(
    values: pd.Series,
    state_labels: pd.Series,
    slot_index: pd.Series,
    neighbour_slots: int = 6
) -> pd.Series:
    unique_states = sorted([x for x in state_labels.dropna().unique()])
    slot_state_medians = {}

    for st in unique_states:
        mask_st = state_labels == st
        values_st = values[mask_st]
        slots_st = slot_index[mask_st]

        by_slot = {
            s: values_st[slots_st == s].dropna().to_numpy()
            for s in range(SLOTS_PER_DAY)
        }

        for s in range(SLOTS_PER_DAY):
            neighbour_values = []

            for off in range(-neighbour_slots, neighbour_slots + 1):
                s2 = (s + off) % SLOTS_PER_DAY
                arr = by_slot[s2]
                if arr.size:
                    neighbour_values.append(arr)

            if neighbour_values:
                slot_state_medians[(st, s)] = float(np.median(np.concatenate(neighbour_values)))
            else:
                slot_state_medians[(st, s)] = np.nan

    keys = list(zip(state_labels, slot_index))
    return pd.Series(
        [slot_state_medians.get(k, np.nan) for k in keys],
        index=values.index,
        dtype=float
    )


# Leakage preparation
# Pipe leakage columns are renamed to leak__<pipe>
# to avoid collisions with flow signal columns p235/p227.
def prepare_leakage(leak_raw: pd.DataFrame):
    leak_cols_original = [c for c in leak_raw.columns if c != TIMESTAMP_COL]

    leak = leak_raw.copy()

    leak["total_leak_flow"] = leak[leak_cols_original].sum(axis=1)
    leak["leak_active"] = (leak["total_leak_flow"] > 0).astype(int)
    leak["active_leak_count"] = (leak[leak_cols_original] > 0).sum(axis=1)
    leak["max_leak_flow"] = leak[leak_cols_original].max(axis=1)

    nonzero = leak.loc[leak["total_leak_flow"] > 0, "total_leak_flow"]
    high_thr = float(nonzero.quantile(HIGH_LEAK_QUANTILE)) if len(nonzero) else 0.0

    leak["high_leak_intensity"] = (
        (leak["total_leak_flow"] >= high_thr) &
        (leak["total_leak_flow"] > 0)
    ).astype(int)

    rename_map = {c: f"leak__{c}" for c in leak_cols_original}
    leak = leak.rename(columns=rename_map)

    leak_cols_prefixed = [rename_map[c] for c in leak_cols_original]
    pipe_to_leak_col = {c: rename_map[c] for c in leak_cols_original}

    return leak, leak_cols_original, leak_cols_prefixed, pipe_to_leak_col, high_thr


# Build timestamp-level flow scores for one year
def build_flow_scores_for_year(year: int) -> pd.DataFrame:
    flows = read_semicolon_csv_robust(FILES[year]["flows"], TIMESTAMP_COL)
    states = read_semicolon_csv_robust(FILES[year]["states"], TIMESTAMP_COL)

    missing_flows = [c for c in FLOW_SIGNALS if c not in flows.columns]
    if missing_flows:
        raise ValueError(
            f"{year}: missing flow columns {missing_flows} in {FILES[year]['flows']}"
        )

    if STATE_COL not in states.columns:
        raise ValueError(
            f"{year}: missing state column '{STATE_COL}' in {FILES[year]['states']}"
        )

    df = flows[[TIMESTAMP_COL] + FLOW_SIGNALS].merge(
        states[[TIMESTAMP_COL, STATE_COL]],
        on=TIMESTAMP_COL,
        how="inner"
    )

    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    df["tod_slot_5min"] = (
        (df[TIMESTAMP_COL].dt.hour * 60 + df[TIMESTAMP_COL].dt.minute) // 5
    ).astype(int)

    new_cols = {}

    for flow_col in FLOW_SIGNALS:
        q = df[flow_col].astype(float)

        baseline_G = pd.Series(float(q.median()), index=df.index, dtype=float)

        baseline_T = compute_tod_median_baseline(
            values=q,
            slot_index=df["tod_slot_5min"],
            neighbour_slots=TOD_NEIGHBOUR_SLOTS
        )

        state_medians = q.groupby(df[STATE_COL]).median().to_dict()
        baseline_S = df[STATE_COL].map(state_medians).astype(float)

        baseline_ST = compute_state_tod_median_baseline(
            values=q,
            state_labels=df[STATE_COL],
            slot_index=df["tod_slot_5min"],
            neighbour_slots=TOD_NEIGHBOUR_SLOTS
        )

        baseline_map = {
            "G": baseline_G,
            "T": baseline_T,
            "S": baseline_S,
            "ST": baseline_ST,
        }

        for b in BASELINES:
            baseline = baseline_map[b]
            dev = q - baseline
            absdev = dev.abs()

            new_cols[f"{flow_col}_baseline_{b}"] = baseline
            new_cols[f"{flow_col}_dev_{b}"] = dev
            new_cols[f"{flow_col}_absdev_{b}"] = absdev

    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    df["year"] = year

    return df


merged_by_year = {}
leak_cols_prefixed_by_year = {}
pipe_to_leak_col_by_year = {}
high_leak_threshold_by_year = {}


for year in YEARS:

    flow_scores = build_flow_scores_for_year(year)

    leak_raw = read_semicolon_csv_robust(FILES[year]["leakages"], TIMESTAMP_COL)
    leak, leak_cols_original, leak_cols_prefixed, pipe_to_leak_col, high_thr = prepare_leakage(leak_raw)

    merged = flow_scores.merge(leak, on=TIMESTAMP_COL, how="inner")
    merged = merged.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    merged_by_year[year] = merged
    leak_cols_prefixed_by_year[year] = leak_cols_prefixed
    pipe_to_leak_col_by_year[year] = pipe_to_leak_col
    high_leak_threshold_by_year[year] = high_thr

    timestamp_out = OUT_DIR / f"flow_timestamp_scores_with_leakage_{year}.csv"
    merged.to_csv(timestamp_out, sep=SEP, index=False)



tail_summary_rows = []

for year in YEARS:
    df = merged_by_year[year]

    global_total_leak_mean = float(df["total_leak_flow"].mean())
    global_high_leak_share = float(df["high_leak_intensity"].mean())
    global_active_leak_count_mean = float(df["active_leak_count"].mean())

    for flow_signal in FLOW_SIGNALS:
        for b in BASELINES:
            score_col = f"{flow_signal}_absdev_{b}"

            if score_col not in df.columns:
                raise ValueError(f"{year}: missing score column {score_col}")

            valid = df[[score_col, "total_leak_flow", "high_leak_intensity", "active_leak_count"]].dropna()
            if valid.empty:
                continue

            for top_pct in TOP_PCTS:
                threshold = valid[score_col].quantile(1 - top_pct)
                tail = df.loc[df[score_col] >= threshold].copy()

                tail_total_leak_mean = float(tail["total_leak_flow"].mean())
                tail_high_leak_share = float(tail["high_leak_intensity"].mean())
                tail_active_leak_count_mean = float(tail["active_leak_count"].mean())

                same_pipe_leak_col = pipe_to_leak_col_by_year[year].get(flow_signal, None)

                if same_pipe_leak_col is not None:
                    tail_same_pipe_leak_sum = float(tail[same_pipe_leak_col].sum())
                    tail_same_pipe_active_share = float((tail[same_pipe_leak_col] > 0).mean())
                else:
                    tail_same_pipe_leak_sum = np.nan
                    tail_same_pipe_active_share = np.nan

                tail_total_leak_sum = float(tail["total_leak_flow"].sum())

                tail_summary_rows.append({
                    "year": year,
                    "flow_signal": flow_signal,
                    "baseline": b,
                    "score_col": score_col,
                    "top_pct": top_pct,
                    "score_threshold": float(threshold),
                    "tail_n": int(len(tail)),

                    "global_total_leak_mean": global_total_leak_mean,
                    "tail_total_leak_mean": tail_total_leak_mean,
                    "tail_total_leak_enrichment_ratio": safe_ratio(
                        tail_total_leak_mean,
                        global_total_leak_mean
                    ),

                    "global_high_leak_share": global_high_leak_share,
                    "tail_high_leak_share": tail_high_leak_share,
                    "tail_high_leak_enrichment_ratio": safe_ratio(
                        tail_high_leak_share,
                        global_high_leak_share
                    ),

                    "global_active_leak_count_mean": global_active_leak_count_mean,
                    "tail_active_leak_count_mean": tail_active_leak_count_mean,
                    "tail_active_leak_count_enrichment_ratio": safe_ratio(
                        tail_active_leak_count_mean,
                        global_active_leak_count_mean
                    ),

                    "same_pipe_available_in_leakage_file": same_pipe_leak_col is not None,
                    "tail_same_pipe_leak_sum": tail_same_pipe_leak_sum,
                    "tail_total_leak_sum": tail_total_leak_sum,
                    "tail_same_pipe_leak_share_of_total": safe_ratio(
                        tail_same_pipe_leak_sum,
                        tail_total_leak_sum
                    ),
                    "tail_same_pipe_active_share": tail_same_pipe_active_share,
                })

tail_summary = pd.DataFrame(tail_summary_rows)

tail_summary_out = OUT_DIR / "flow_top_tail_leakage_summary_2018_vs_2019.csv"
tail_summary.to_csv(tail_summary_out, sep=SEP, index=False)

print("\nTop-tail leakage summary:")
display(tail_summary)


tail_pipe_rows = []

for year in YEARS:
    df = merged_by_year[year]
    leak_cols_prefixed = leak_cols_prefixed_by_year[year]

    for flow_signal in FLOW_SIGNALS:
        for b in BASELINES:
            score_col = f"{flow_signal}_absdev_{b}"

            if score_col not in df.columns:
                raise ValueError(f"{year}: missing score column {score_col}")

            for top_pct in TOP_PCTS:
                threshold = df[score_col].quantile(1 - top_pct)
                tail = df.loc[df[score_col] >= threshold].copy()

                tail_total_leak_sum = float(tail["total_leak_flow"].sum())

                pipe_sums = tail[leak_cols_prefixed].sum(axis=0).sort_values(ascending=False)
                pipe_active_share = (tail[leak_cols_prefixed] > 0).mean(axis=0).reindex(pipe_sums.index)

                for rank, leak_col in enumerate(pipe_sums.index, start=1):
                    pipe = leak_col.replace("leak__", "", 1)
                    pipe_sum = float(pipe_sums.loc[leak_col])

                    tail_pipe_rows.append({
                        "year": year,
                        "flow_signal": flow_signal,
                        "baseline": b,
                        "score_col": score_col,
                        "top_pct": top_pct,
                        "score_threshold": float(threshold),
                        "tail_n": int(len(tail)),

                        "pipe_rank_in_tail": rank,
                        "pipe": pipe,
                        "pipe_leak_sum_in_tail": pipe_sum,
                        "pipe_share_of_tail_total_leak": safe_ratio(
                            pipe_sum,
                            tail_total_leak_sum
                        ),
                        "pipe_active_share_in_tail": float(pipe_active_share.loc[leak_col]),
                        "is_same_as_flow_signal": pipe == flow_signal,
                    })

tail_pipe_composition = pd.DataFrame(tail_pipe_rows)

tail_pipe_out = OUT_DIR / "flow_top_tail_pipe_composition_2018_vs_2019.csv"
tail_pipe_composition.to_csv(tail_pipe_out, sep=SEP, index=False)


print("\nTop 15 leaking pipes inside each top-tail subset:")
display(
    tail_pipe_composition
    .query("pipe_rank_in_tail <= 15")
    .sort_values(["year", "flow_signal", "baseline", "top_pct", "pipe_rank_in_tail"])
)


local_vs_network_rows = []

for year in YEARS:
    df = merged_by_year[year]
    pipe_to_leak_col = pipe_to_leak_col_by_year[year]

    for flow_signal in FLOW_SIGNALS:
        same_pipe_leak_col = pipe_to_leak_col.get(flow_signal, None)

        for b in BASELINES:
            score_col = f"{flow_signal}_absdev_{b}"

            valid = df[[score_col, "total_leak_flow", "high_leak_intensity", "active_leak_count"]].copy()

            if same_pipe_leak_col is not None:
                valid[same_pipe_leak_col] = df[same_pipe_leak_col]

            valid = valid.dropna(subset=[score_col])

            low_thr = valid[score_col].quantile(0.20)
            high_thr = valid[score_col].quantile(0.80)

            low_score = valid.loc[valid[score_col] <= low_thr]
            high_score = valid.loc[valid[score_col] >= high_thr]

            network_low_mean = float(low_score["total_leak_flow"].mean())
            network_high_mean = float(high_score["total_leak_flow"].mean())

            low_high_leak_share = float(low_score["high_leak_intensity"].mean())
            high_high_leak_share = float(high_score["high_leak_intensity"].mean())

            row = {
                "year": year,
                "flow_signal": flow_signal,
                "baseline": b,
                "score_col": score_col,

                "low_score_threshold_q20": float(low_thr),
                "high_score_threshold_q80": float(high_thr),
                "low_score_n": int(len(low_score)),
                "high_score_n": int(len(high_score)),

                "network_low_total_leak_mean": network_low_mean,
                "network_high_total_leak_mean": network_high_mean,
                "network_high_low_total_leak_mean_ratio": safe_ratio(
                    network_high_mean,
                    network_low_mean
                ),

                "low_high_leak_share": low_high_leak_share,
                "high_high_leak_share": high_high_leak_share,
                "network_high_low_high_leak_share_ratio": safe_ratio(
                    high_high_leak_share,
                    low_high_leak_share
                ),

                "low_active_leak_count_mean": float(low_score["active_leak_count"].mean()),
                "high_active_leak_count_mean": float(high_score["active_leak_count"].mean()),

                "same_pipe_available_in_leakage_file": same_pipe_leak_col is not None,
            }

            if same_pipe_leak_col is not None:
                local_low_mean = float(low_score[same_pipe_leak_col].mean())
                local_high_mean = float(high_score[same_pipe_leak_col].mean())

                row.update({
                    "local_same_pipe_low_leak_mean": local_low_mean,
                    "local_same_pipe_high_leak_mean": local_high_mean,
                    "local_same_pipe_high_low_mean_ratio": safe_ratio(
                        local_high_mean,
                        local_low_mean
                    ),
                    "local_same_pipe_low_active_share": float((low_score[same_pipe_leak_col] > 0).mean()),
                    "local_same_pipe_high_active_share": float((high_score[same_pipe_leak_col] > 0).mean()),
                })
            else:
                row.update({
                    "local_same_pipe_low_leak_mean": np.nan,
                    "local_same_pipe_high_leak_mean": np.nan,
                    "local_same_pipe_high_low_mean_ratio": np.nan,
                    "local_same_pipe_low_active_share": np.nan,
                    "local_same_pipe_high_active_share": np.nan,
                })

            local_vs_network_rows.append(row)

local_vs_network = pd.DataFrame(local_vs_network_rows)

local_vs_network_out = OUT_DIR / "flow_local_vs_network_leakage_check_2018_vs_2019.csv"
local_vs_network.to_csv(local_vs_network_out, sep=SEP, index=False)

print("\nLocal same-pipe vs network leakage comparison:")
display(local_vs_network)


print("\nCompact top-tail enrichment summary:")
compact_cols = [
    "year", "flow_signal", "baseline", "top_pct",
    "tail_total_leak_enrichment_ratio",
    "tail_high_leak_enrichment_ratio",
    "tail_active_leak_count_enrichment_ratio",
    "same_pipe_available_in_leakage_file",
]
display(
    tail_summary[compact_cols]
    .sort_values(["flow_signal", "baseline", "top_pct", "year"])
)

print("\nTop 5 pipes in top 1% ST tail:")
display(
    tail_pipe_composition
    .query("baseline == 'ST' and top_pct == 0.01 and pipe_rank_in_tail <= 5")
    .sort_values(["year", "flow_signal", "pipe_rank_in_tail"])
)


Top-tail leakage summary:


,year,flow_signal,baseline,score_col,top_pct,score_threshold,tail_n,global_total_leak_mean,tail_total_leak_mean,tail_total_leak_enrichment_ratio,...,tail_high_leak_share,tail_high_leak_enrichment_ratio,global_active_leak_count_mean,tail_active_leak_count_mean,tail_active_leak_count_enrichment_ratio,same_pipe_available_in_leakage_file,tail_same_pipe_leak_sum,tail_total_leak_sum,tail_same_pipe_leak_share_of_total,tail_same_pipe_active_share
0,2018,p235,G,p235_absdev_G,0.01,81.89930,1052,25.526981,6.622757,0.259441,...,0.000951,0.004859,3.448111,1.634981,0.474167,False,NaN,6967.14,NaN,NaN
1,2018,p235,G,p235_absdev_G,0.05,67.85000,5259,25.526981,18.284583,0.716285,...,0.051341,0.262441,3.448111,2.932116,0.850355,False,NaN,96158.62,NaN,NaN
2,2018,p235,T,p235_absdev_T,0.01,48.63000,1053,25.526981,63.781529,2.498593,...,0.808167,4.131156,3.448111,3.389364,0.982963,False,NaN,67161.95,NaN,NaN
3,2018,p235,T,p235_absdev_T,0.05,36.63650,5256,25.526981,46.189243,1.809428,...,0.564688,2.886549,3.448111,2.966514,0.860330,False,NaN,242770.66,NaN,NaN
4,2018,p235,S,p235_absdev_S,0.01,79.90000,1053,25.526981,7.641690,0.299357,...,0.001899,0.009709,3.448111,1.783476,0.517233,False,NaN,8046.70,NaN,NaN
5,2018,p235,S,p235_absdev_S,0.05,63.03000,5258,25.526981,21.538636,0.843760,...,0.118676,0.606645,3.448111,3.018448,0.875392,False,NaN,113250.15,NaN,NaN
6,2018,p235,ST,p235_absdev_ST,0.01,48.22465,1052,25.526981,62.860618,2.462517,...,0.793726,4.057338,3.448111,3.345057,0.970113,False,NaN,66129.37,NaN,NaN
7,2018,p235,ST,p235_absdev_ST,0.05,36.10000,5258,25.526981,47.521936,1.861636,...,0.580068,2.965171,3.448111,3.045264,0.883169,False,NaN,249870.34,NaN,NaN
8,2018,p227,G,p227_absdev_G,0.01,79.86000,1054,25.526981,6.932125,0.271561,...,0.000000,0.000000,3.448111,1.697343,0.492253,False,NaN,7306.46,NaN,NaN
9,2018,p227,G,p227_absdev_G,0.05,64.62300,5256,25.526981,16.396703,0.642328,...,0.020167,0.103091,3.448111,2.802702,0.812822,False,NaN,86181.07,NaN,NaN


Top 15 leaking pipes inside each top-tail subset:


,year,flow_signal,baseline,score_col,top_pct,score_threshold,tail_n,pipe_rank_in_tail,pipe,pipe_leak_sum_in_tail,pipe_share_of_tail_total_leak,pipe_active_share_in_tail,is_same_as_flow_signal
112,2018,p227,G,p227_absdev_G,0.01,79.86,1054,1,p257,4996.78,0.683885,0.873814,False
113,2018,p227,G,p227_absdev_G,0.01,79.86,1054,2,p461,1496.37,0.204801,0.365275,False
114,2018,p227,G,p227_absdev_G,0.01,79.86,1054,3,p427,740.91,0.101405,0.414611,False
115,2018,p227,G,p227_absdev_G,0.01,79.86,1054,4,p232,44.28,0.006060,0.033207,False
116,2018,p227,G,p227_absdev_G,0.01,79.86,1054,5,p31,18.53,0.002536,0.002846,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,2019,p235,T,p235_absdev_T,0.05,42.40,5257,11,p193,1064.44,0.006303,0.047556,False
304,2019,p235,T,p235_absdev_T,0.05,42.40,5257,12,p514,729.32,0.004318,0.008940,False
305,2019,p235,T,p235_absdev_T,0.05,42.40,5257,13,p277,605.74,0.003587,0.035952,False
306,2019,p235,T,p235_absdev_T,0.05,42.40,5257,14,p721,601.72,0.003563,0.012174,False


Local same-pipe vs network leakage comparison:


,year,flow_signal,baseline,score_col,low_score_threshold_q20,high_score_threshold_q80,low_score_n,high_score_n,network_low_total_leak_mean,network_high_total_leak_mean,...,high_high_leak_share,network_high_low_high_leak_share_ratio,low_active_leak_count_mean,high_active_leak_count_mean,same_pipe_available_in_leakage_file,local_same_pipe_low_leak_mean,local_same_pipe_high_leak_mean,local_same_pipe_high_low_mean_ratio,local_same_pipe_low_active_share,local_same_pipe_high_active_share
0,2018,p235,G,p235_absdev_G,7.30,40.490,21037,21025,21.836853,26.873483,...,0.214316,1.936672,3.284118,3.315149,False,NaN,NaN,NaN,NaN,NaN
1,2018,p235,T,p235_absdev_T,3.61,19.690,21034,21043,23.378472,30.724686,...,0.340636,3.780968,3.685652,3.091527,False,NaN,NaN,NaN,NaN,NaN
2,2018,p235,S,p235_absdev_S,6.78,37.880,21031,21028,22.232929,27.420155,...,0.226745,2.032686,3.332842,3.336504,False,NaN,NaN,NaN,NaN,NaN
3,2018,p235,ST,p235_absdev_ST,3.45,19.045,21022,21026,23.167573,31.122058,...,0.351803,4.235735,3.680620,3.107438,False,NaN,NaN,NaN,NaN,NaN
4,2018,p227,G,p227_absdev_G,7.32,38.050,21026,21026,21.633750,23.542166,...,0.162608,1.582138,3.200323,3.280795,False,NaN,NaN,NaN,NaN,NaN
5,2018,p227,T,p227_absdev_T,3.82,19.000,21024,21025,23.645650,25.125138,...,0.251510,2.500117,3.576769,3.134887,False,NaN,NaN,NaN,NaN,NaN
6,2018,p227,S,p227_absdev_S,6.81,35.750,21024,21025,22.117176,24.128674,...,0.175838,1.652581,3.247479,3.322854,False,NaN,NaN,NaN,NaN,NaN
7,2018,p227,ST,p227_absdev_ST,3.69,18.480,21040,21023,23.390788,25.368701,...,0.258717,2.714912,3.564163,3.155211,False,NaN,NaN,NaN,NaN,NaN
8,2019,p235,G,p235_absdev_G,10.74,41.200,21031,21023,68.594692,65.298043,...,0.146506,1.263811,9.533023,9.146506,False,NaN,NaN,NaN,NaN,NaN
9,2019,p235,T,p235_absdev_T,4.59,25.680,21025,21026,78.838198,48.399200,...,0.107534,0.978741,11.111629,7.014744,False,NaN,NaN,NaN,NaN,NaN



Compact top-tail enrichment summary:


,year,flow_signal,baseline,top_pct,tail_total_leak_enrichment_ratio,tail_high_leak_enrichment_ratio,tail_active_leak_count_enrichment_ratio,same_pipe_available_in_leakage_file
8,2018,p227,G,0.01,0.271561,0.000000,0.492253,False
24,2019,p227,G,0.01,0.412141,0.000000,0.468875,False
9,2018,p227,G,0.05,0.642328,0.103091,0.812822,False
25,2019,p227,G,0.05,0.561465,0.040876,0.598732,False
12,2018,p227,S,0.01,0.271289,0.000000,0.497876,False
28,2019,p227,S,0.01,0.400263,0.000000,0.507844,False
13,2018,p227,S,0.05,0.656013,0.228464,0.810031,False
29,2019,p227,S,0.05,0.570288,0.109319,0.633176,False
14,2018,p227,ST,0.01,0.406116,0.757298,0.520813,False
30,2019,p227,ST,0.01,0.710725,1.021504,0.743246,False



Top 5 pipes in top 1% ST tail:


,year,flow_signal,baseline,score_col,top_pct,score_threshold,tail_n,pipe_rank_in_tail,pipe,pipe_leak_sum_in_tail,pipe_share_of_tail_total_leak,pipe_active_share_in_tail,is_same_as_flow_signal
196,2018,p227,ST,p227_absdev_ST,0.01,40.88000,1053,1,p257,3113.65,0.285228,0.690408,False
197,2018,p227,ST,p227_absdev_ST,0.01,40.88000,1053,2,p31,2406.65,0.220463,0.157645,False
198,2018,p227,ST,p227_absdev_ST,0.01,40.88000,1053,3,p183,2230.76,0.204350,0.131054,False
199,2018,p227,ST,p227_absdev_ST,0.01,40.88000,1053,4,p427,1036.33,0.094934,0.293447,False
200,2018,p227,ST,p227_absdev_ST,0.01,40.88000,1053,5,p461,729.78,0.066852,0.169041,False
84,2018,p235,ST,p235_absdev_ST,0.01,48.22465,1052,1,p628,28594.57,0.432403,0.782319,False
85,2018,p235,ST,p235_absdev_ST,0.01,48.22465,1052,2,p538,26865.55,0.406257,0.788973,False
86,2018,p235,ST,p235_absdev_ST,0.01,48.22465,1052,3,p257,5988.63,0.090559,0.909696,False
87,2018,p235,ST,p235_absdev_ST,0.01,48.22465,1052,4,p427,4187.81,0.063328,0.806084,False
88,2018,p235,ST,p235_absdev_ST,0.01,48.22465,1052,5,p866,163.49,0.002472,0.007605,False
